## 🔄 Restore Session
> **Run this cell FIRST after any crash or Colab/Kaggle reset.**  
> Mounts Drive, restores checkpoint, re-clones repo if needed, reinstalls deps.  
> All subsequent cells will resume from where you left off.


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 🔄 RESTORE SESSION — run this FIRST after any crash or runtime reset
# ══════════════════════════════════════════════════════════════════════
import os, sys, subprocess
from pathlib import Path

IS_COLAB  = 'google.colab' in sys.modules or os.path.exists('/content')
IS_KAGGLE = os.path.exists('/kaggle')
GITHUB_REPO = "LLM-HypatiaX-PAPERS-Public"

# ── 1. Mount persistent storage (Colab) ──────────────────────────────
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PERSISTENT = Path('/content/drive/MyDrive/HypatiaX')
    PERSISTENT.mkdir(parents=True, exist_ok=True)
    REPO = Path('/content') / GITHUB_REPO
    print(f'Drive mounted. Persistent storage: {PERSISTENT}')
elif IS_KAGGLE:
    PERSISTENT = Path('/kaggle/working/HypatiaX_persistent')
    PERSISTENT.mkdir(parents=True, exist_ok=True)
    REPO = Path('/kaggle/working') / GITHUB_REPO
    print('Kaggle: persistent dir at', PERSISTENT)
else:
    PERSISTENT = Path.home() / 'HypatiaX_persistent'
    PERSISTENT.mkdir(parents=True, exist_ok=True)
    REPO = Path('..') / GITHUB_REPO
    print('Local: persistent dir at', PERSISTENT)

# ── 2. Restore checkpoint from persistent storage ────────────────────
CP_PERSISTENT = PERSISTENT / 'pipeline_checkpoint.json'
CP_LOGS       = REPO / 'logs' / 'pipeline_checkpoint.json'
CP_NB         = REPO / 'hypatiax_checkpoint.json'

if CP_PERSISTENT.exists():
    # Copy to all locations the pipeline looks for it
    CP_LOGS.parent.mkdir(parents=True, exist_ok=True)
    import shutil
    shutil.copy2(CP_PERSISTENT, CP_LOGS)
    shutil.copy2(CP_PERSISTENT, CP_NB)
    import json
    cp = json.loads(CP_PERSISTENT.read_text())
    done = cp.get('completed', [])
    print(f'✅ Checkpoint restored: {len(done)} experiment(s) already done: {done}')
else:
    print('ℹ️  No checkpoint found — starting fresh')

# ── 3. Clone repo if missing ─────────────────────────────────────────
if not REPO.exists():
    url = f'https://github.com/sednabcn/{GITHUB_REPO}.git'
    print(f'Cloning {url} ...')
    r = subprocess.run(['git', 'clone', url, str(REPO)], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'Clone failed:\n{r.stderr}')
    print(f'✅ Cloned to {REPO}')
else:
    print(f'✅ Repo already present: {REPO}')

# ── 4. Reinstall dependencies (pip cache makes this fast after first run) ──
_flag = PERSISTENT / '.deps_installed'
if _flag.exists():
    print('✅ Dependencies already installed (cached)')
else:
    print('Installing dependencies ...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '-r', str(REPO / 'requirements.txt')], check=False)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'gitpython==3.1.47'], check=False)
    _flag.touch()
    print('✅ Dependencies installed')

# ── 5. Set REPO env var so all downstream cells find it ──────────────
os.environ['REPO_ROOT_PATH'] = str(REPO.resolve())
os.environ['REPRO_ROOT']     = str(REPO.resolve())
sys.path.insert(0, str(REPO.resolve()))
os.chdir(str(REPO.resolve()))
print(f'✅ CWD → {os.getcwd()}')

# ── 6. Set PERSISTENT_DIR so save_results() knows where to back up ───
os.environ['PERSISTENT_DIR'] = str(PERSISTENT)
print(f'✅ PERSISTENT_DIR = {PERSISTENT}')
print()
print('══════════════════════════════════════════════')
print('  Session restored — run setup cells next')
print('  (API key → Runtime Config → patches → exp)')
print('══════════════════════════════════════════════')


## 🔁 Per-Case Restore After Crash
> **Dead during normal execution** — run this cell right after the Restore Session cell every time you reconnect.  
> It compares case result files between persistent storage and the working directory.  
> If nothing is missing it prints one line and exits instantly.  
> If a crash left results behind it copies only the missing files, merges logs, and restores the checkpoint.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 🔁  PER-CASE RESTORE — dead during normal execution
#     Run after the Restore Session cell whenever you reconnect.
#     Copies any case results saved in persistent storage that are
#     missing from the working directory.  No-op if nothing is missing.
# ══════════════════════════════════════════════════════════════════════
import os, shutil, json, pathlib, datetime

# ── Derive paths (works even if env vars are not yet set) ────────────
IS_COLAB  = 'google.colab' in __import__('sys').modules or pathlib.Path('/content').exists()
IS_KAGGLE = pathlib.Path('/kaggle').exists()
GITHUB_REPO = "LLM-HypatiaX-PAPERS-Public"

if IS_COLAB:
    PERSISTENT = pathlib.Path('/content/drive/MyDrive/HypatiaX')
    REPO       = pathlib.Path('/content') / GITHUB_REPO
elif IS_KAGGLE:
    PERSISTENT = pathlib.Path('/kaggle/working/HypatiaX_persistent')
    REPO       = pathlib.Path('/kaggle/working') / GITHUB_REPO
else:
    PERSISTENT = pathlib.Path.home() / 'HypatiaX_persistent'
    REPO       = pathlib.Path('..') / GITHUB_REPO

# Honour env overrides if Restore Session already ran
PERSISTENT = pathlib.Path(os.environ.get('PERSISTENT_DIR', str(PERSISTENT)))
REPO       = pathlib.Path(os.environ.get('REPRO_ROOT',    str(REPO)))

PERS_RESULTS = PERSISTENT / 'results'
WORK_RESULTS = REPO / 'hypatiax' / 'data' / 'results'
PERS_LOGS    = PERSISTENT / 'logs'
WORK_LOGS    = REPO / 'logs'
PERS_CP      = PERSISTENT / 'pipeline_checkpoint.json'
WORK_CP      = REPO / 'logs' / 'pipeline_checkpoint.json'

# ── Count case files on each side ────────────────────────────────────
def _count(d):
    return len(list(d.rglob('*.json'))) if d.exists() else 0

pers_n = _count(PERS_RESULTS)
work_n = _count(WORK_RESULTS)

# ── Guard: nothing to restore ─────────────────────────────────────────
if pers_n == 0:
    print('✅ Normal session — persistent storage has no case results yet')
elif work_n >= pers_n:
    print(f'✅ Normal session — working dir has {work_n} case files'
          f' (≥ persistent {pers_n}) — nothing to restore')
else:
    # ── Crash detected: restore everything ───────────────────────────
    missing = pers_n - work_n
    print(f'⚠️  Post-crash restore: {work_n} local / {pers_n} persistent'
          f' → copying {missing} missing case file(s) ...')

    copied, skipped, errors = 0, 0, []

    # 1. Per-case result files
    WORK_RESULTS.mkdir(parents=True, exist_ok=True)
    for src in PERS_RESULTS.rglob('*.json'):
        rel = src.relative_to(PERS_RESULTS)
        dst = WORK_RESULTS / rel
        if dst.exists():
            skipped += 1
        else:
            try:
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src, dst)
                copied += 1
            except Exception as e:
                errors.append(f'{rel}: {e}')

    # 2. Logs directory (non-destructive merge)
    if PERS_LOGS.exists():
        try:
            shutil.copytree(PERS_LOGS, WORK_LOGS, dirs_exist_ok=True)
        except Exception as e:
            errors.append(f'logs: {e}')

    # 3. Checkpoint file
    if PERS_CP.exists():
        try:
            WORK_CP.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(PERS_CP, WORK_CP)
            cp = json.loads(PERS_CP.read_text())
            done = cp.get('completed', [])
            print(f'  ✓ checkpoint: {len(done)} completed experiment(s): {done}')
        except Exception as e:
            errors.append(f'checkpoint: {e}')

    # 4. Summary
    print(f'  ✓ results:    {copied} copied, {skipped} already present')
    if errors:
        print(f'  ⚠ errors ({len(errors)}): {errors}')
    else:
        ts = datetime.datetime.now().strftime('%H:%M:%S')
        print(f'✅ [{ts}] Restore complete — resume experiments from where you left off')


# HypatiaX — Unified Experiment Notebook  v10
**Paper:** *HypatiaX: A Hybrid Symbolic-Neural Framework for Extrapolation-Reliable Analytical Discovery*  
**Author:** Ruperto Pedro Bonet Chaple · **Venue:** JMLR v3.0, April 2026  
**LLM component:** Claude (`claude-sonnet-4-20250514`) via Anthropic API — LLM results are stochastic  
**NN / PySR:** exactly reproducible at seed 42

> **v8 changes (synced with run_all_checkpoint.py v4.8):** exp1b FIX-EXP1B-ARGS — no `--task`/`--seeds` CLI args, env vars only; exp3/exp3b use `-m hypatiax.protocols.experiment_protocol_nguyen12_exp3` module call; exp3b runs all 4 stability seeds (99/123/777/2024); Supp A/B/exp1/exp2 use `hypatiax/protocols/` prefix matching run_all step defs; K=30 hardcoded in instability env; `! python run_all_checkpoint.py --only <exp>` convenience comment cells added before every experiment.  
> **v7 changes (synced with run_all_checkpoint.py v4.8):** `claude-sonnet-4-20250514` canonical model string; `ONE_EQUATION` / `--one-equation` smoke-test wired to `FAST=1` branch; `fixup-init` step added to 0-C setup; `N_TASKS_INSTABILITY` corrected to 70 (FIX-T1); `DEFI_V3C_NO_TIMEOUT_FLAGS` / `DEFI_TASK_FILTER` / `DEFI_SEEDS` / `SKIP_PKG_CHECK` / `HYPATIAX_CORE_OPTIONAL` env vars added where matching run_all steps set them; `VERIFY_RESULTS_DIR` / `PATCHED_DATA_DIR` propagated to verify cell (FIX-VERIFY); `TABLE_OUTDIR` propagated to tables cell (FIX-TABLES); NB-06 pre-audit block aligned with run_all.

| # | Experiment | Protocol | Paper section | Expected result |
|---|---|---|---|---|
| Exp 1  | DeFi 74-task benchmark v3.0      | `experiment_protocol_ablation_exp1.py`             | §10.2–10.4, §10.6 | 89.2 % R²>0.99 · 0 catastrophic · 1.73× speedup |
| Exp 1b | Portfolio Variance seed sweep     | `experiment_protocol_defi_v3.py`                   | §10.5             | P(H>P) ≈ 0.76 across seeds 42/99/123/777/2024   |
| Exp 2  | Feynman 30-equation extrapolation | `experiment_protocol_feynman_exp2.py`              | §10.7             | 9/30 (30 %)  *(4–8 h)*                          |
| Exp 3  | Nguyen-12 SR suite (seed 42)      | `experiment_protocol_nguyen12_exp3.py`             | §10.8             | 11/12 H (91.7 %)                                |
| Exp 3b | Nguyen-12 stability (seeds 99/123/777/2024)    | `experiment_protocol_nguyen12_exp3.py --seed 123`  | §10.8 stability   | consistent with seed 42                         |
| Supp B | Noise/sample-complexity sweep     | `experiment_protocol_noise_sweep.py`               | Supp B            | EHD 100 % at all σ · plateau ≈ N=500            |
| Supp A | Hybrid routing improvements       | `experiment_protocol_hybrid_routing.py`            | Supp A §1–8       | +6 pp Fix1, +5 pp Fix2, +1 pp Fix3              |
| §10.9  | Instability analysis (K=30)       | `experiment_protocol_instability_rf02_04.py`       | §10.9             | Spearman ρ=−0.70, p<0.001                       |
| §10.8c | Extrapolation comparative         | `experiment_protocol_extrapolation_comparative.py` | §10.8             | cross-method R² across OOD regimes              |
| §11a   | Provenance audit (orchestration)  | `experiment_protocol_provenance_audit.py`          | §11               | run after all experiments                       |
| §11b   | Result-family linker              | `audit/discover_provenance.py`                     | §11               | links result files → families → paper tables    |
| §11c   | Internal import DAG               | `audit/scan_internal_imports.py`                   | §11               | logs/repro_output/import_graph.dot              |

| §12a   | Bib audit                         | `NB-01_Citation_Bibliography_Audit.ipynb`          | §B    | FIX-B1/B2/B3 — missing/dup bibitems            |
| §12b   | Cross-reference audit              | `NB-02_CrossReference_Label_Audit.ipynb`           | §B    | FIX-XR1–XR4 — undef refs, dup labels            |
| §12c   | Section structure audit            | `NB-03_Section_Structure_Numbering.ipynb`          | §B    | diagnostic — section tree & equation count      |
| §12d   | Numerical consistency audit        | `NB-04_Numerical_Consistency_Checker.ipynb`        | §B    | FIX-N1/N2/N3 — 70 vs 71, terminology           |
| §12e   | Figure dependency audit            | `NB-05_Figure_Image_Dependency_Checker.ipynb`      | §B    | FIX-F1–F4 — missing figures, fbox placeholders |
| §12f   | Code quality pre-audit             | `NB-06_Code_Quality_Pipeline_Integrity.ipynb`      | §B    | FIX-C1/C3 — dup case names, split mismatch     |

> **Run order:** cells are ordered to match the pipeline. Run top-to-bottom once.  
> For Colab: skip SSH cells (1-A / 1-B / Step 2) if using the public repo.


> **PAPER-QUALITY VERSION** — `FAST=0` hardcoded. Full run ~15-25 h. Colab Pro+ or Kaggle + skip-slow recommended.


## Platform Setup — Colab / Kaggle
> Run this cell **first**, before any other cell. Sets `FAST=0` (paper-quality) and mounts Drive on Colab.


## Fixes applied (v9 → v10)

| # | File | Fix |
|---|------|-----|
| 1 | `hypatiax/core/metrics.py` | Created missing module — `compute_r2` now importable |
| 2 | `experiment_protocol_benchmark.py` | Moved `from __future__ import annotations` to line 1 |
| 3 | `exp2_feynman_colab_multithreaded.py` | Guarded bare `import pysr` with try/except |
| 4 | `hypatiax_exp1_ablation.py` | Guarded bare `import pysr` + `import juliacall` |
| 5 | 9× training/generation files | Guarded `import torch` + added `nn` stub for class definitions |
| 6 | `references.bib` | Added 9 missing `\cite` keys |
| 7 | `jmlr-hypatiax-paper-final.tex` | Fixed duplicate `\label`, `\label` inside `\item`, `\ref` targets, `71 cases`→`70 tasks`, 5× `five-stage routing`→`Five-Layer Architecture` |
| 8 | `figures/` | Created 5 placeholder figures (PNG+PDF) |
| 9 | `NB-01–06` | Fixed bib parser, path prefixes, section-label regex, cell formats — all 6 now **PASS** |

| 10 | `run_all_checkpoint.py` | Removed `/`-reaching `sys.executable.parent.parent.parent` from `_julia_roots`; added `_BLOCKED` set + `OSError` guard on both `rglob` sites in `_clear_stale_locks()` — fixes crash on Colab where that path resolves to `/` and walks `/proc` |
| 11 | `hypatiax/core/metrics.py` | Replaced placeholder import redirect with a proper module: `compute_r2`, `compute_mse`, `compute_rmse`, `compute_mae`, `compute_max_error`, `compute_near_perfect_rate`, `compute_catastrophic_rate`, `compute_speedup`, `evaluate_all` — fully tested against sklearn |
| 12 | Experiment run cells (33/38/41/44/47/56) | Added `os.environ['METHOD_TIMEOUT']='900'` + `os.environ['PYSR_TIMEOUT']='1100'` guards before every `run_all_checkpoint.py` call — prevents stale `METHOD_TIMEOUT=1800` from triggering the 300s cap → 630s wall-clock that kills PySR before its own 1100s timeout |
| 13 | New patch cell (after FIX-LOCKS) | `FIX-WALLCLOCK` — patches `exp1_ablation_populations_30_updated.py` wall-clock formula from `timeout_s * 2.1` to `max(timeout_s, PYSR_TIMEOUT) * 1.15` so the wall-clock always covers the full PySR run |

> **Julia/PySR note:** `exp1`, `exp2`, `exp3`, `exp3b`, `benchmark`, `benchmark_v2` require Julia.
> Julia cannot be downloaded in restricted sandboxes (julialang-s3.julialang.org blocked).
> Run on **Colab or Kaggle** where Julia installs automatically on first PySR use.


In [ ]:
# ── Platform detection & FAST=0 override ─────────────────────────────────
import os, sys

# PAPER-QUALITY: force FAST=0 unconditionally
os.environ['FAST'] = '0'

# Seeds (do not change for paper reproducibility)
os.environ.setdefault('NN_SEED',    '42')
os.environ.setdefault('PYSR_SEED',  '42')
os.environ.setdefault('PYTHONHASHSEED', '42')

# Detect platform
IS_COLAB  = 'google.colab' in sys.modules or os.path.exists('/content')
IS_KAGGLE = os.path.exists('/kaggle')

if IS_COLAB:
    print('Platform : Google Colab')
    print('Drive already mounted by Restore Session cell')
    os.system('pip install -q --upgrade pip')
elif IS_KAGGLE:
    print('Platform : Kaggle')
    print('NOTE     : 12-hour hard limit — session is hard-killed at exactly 12 h.')
    # PAPER-QUALITY on Kaggle: all experiments run across multiple sessions.
    # checkpoint/resume (cell below) tracks what is done — re-run the notebook
    # from the top each new session; completed experiments are skipped automatically.
    # ── Output persistence reminder ───────────────────────────────────────
    print()
    print('⚠  KAGGLE OUTPUT: results live only in /kaggle/working/.')
    print('   Before the session ends, go to Data > Output and click')
    print('   "Save Version" (or run the Download Results cell) to keep them.')
else:
    print('Platform : Local / other')


print(f'FAST     : {os.environ["FAST"]} (0 = paper-quality)')
print(f'Seed     : NN={os.environ["NN_SEED"]} / PYSR={os.environ["PYSR_SEED"]}')


Platform : Google Colab
Mounted at /content/drive
Drive    : mounted at /content/drive
FAST     : 0 (0 = paper-quality)
Seed     : NN=42 / PYSR=42


## GPU / Resource Check
> Verifies available accelerator. PySR benefits from more CPU cores; GPU helps NN training.


In [ ]:
import os, multiprocessing

print(f'CPU cores : {multiprocessing.cpu_count()}')

try:
    mem_raw = !grep MemTotal /proc/meminfo | awk '{print $2}'
    mem = mem_raw[0].strip() if mem_raw else ''
    gb = round(int(mem) / 1024**2, 1) if mem else '?'
    print(f'RAM (GB)  : {gb}')
except:
    print('RAM       : unknown')

try:
    gpu_raw = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null
    gpu = gpu_raw[0].strip() if gpu_raw else ''
    print(f'GPU       : {gpu}' if gpu else 'GPU       : none (CPU-only — PySR slower)')
except:
    print('GPU       : check skipped')

print('Tip: Colab → Runtime > Change runtime type > T4 GPU')
print('     Kaggle → right panel > Session options > Accelerator > GPU T4 x2')


CPU cores : 2
RAM (GB)  : 12.7
GPU       : Tesla T4, 15360 MiB
Tip: Colab → Runtime > Change runtime type > T4 GPU
     Kaggle → right panel > Session options > Accelerator > GPU T4 x2


## 0-A · Clone repo & install dependencies
> Run the code cell immediately below to clone **`LLM-HypatiaX-PAPERS-Public`**
> via plain HTTPS (no GitHub account needed) and install all Python dependencies.
> If the repo is already present locally, the cell is a no-op.
>
> The three Raw cells that follow (1-A / 1-B / Step 2) contain the original SSH
> setup for the private repo. **Public-repo users: leave them as Raw and skip to 0-B.**


## 🌐 Public repo — no SSH or GitHub account required

> This notebook targets **`sednabcn/LLM-HypatiaX-PAPERS-Public`** — a fully
> public repository. No SSH key, no GitHub account, and no Personal Access
> Token are needed.

**What to do:**

1. **Skip cells 1-A, 1-B, and Step 2** (SSH key setup — not needed for the public repo).
2. Run the **0-A clone cell** below to download the repository via plain HTTPS.
3. Continue from **0-B** onward.

---
> **Private-repo users (JMLR reviewers with collaborator access):** the three
> Raw cells below (1-A · 1-B · Step 2) contain the original SSH setup. Convert
> them to Code, follow the four-step instructions printed inside, then replace
> `PUBLIC_REPO` in the clone cell with `LLM-HypatiaX-PAPERS` and re-run.


### 1-A · Generate key &nbsp;⬆️ &nbsp;&nbsp;|&nbsp;&nbsp; 1-B · Restore key &nbsp;⬇️
> **Every session:** run only **1-B**. Run **1-A** once if you have no key yet.


In [ ]:
# ── 0-A · Clone public repo (HTTPS, no authentication required) ──────────
import os, sys
from pathlib import Path

GITHUB_USER = "sednabcn"
GITHUB_REPO = "LLM-HypatiaX-PAPERS-Public"   # public — no token needed

# ── Platform-aware clone destination ─────────────────────────────────────
IS_COLAB  = "google.colab" in sys.modules or os.path.exists("/content")
IS_KAGGLE = os.path.exists("/kaggle")

if IS_COLAB:
    src = Path("/content") / GITHUB_REPO
elif IS_KAGGLE:
    src = Path("/kaggle/working") / GITHUB_REPO
else:
    src = Path.cwd().parent / GITHUB_REPO   # local: sibling of notebook directory

# ── Kaggle internet check — must be enabled in Session options ───────────
if IS_KAGGLE:
    !git ls-remote --exit-code --heads https://github.com/sednabcn/LLM-HypatiaX-PAPERS-Public.git
    if _exit_code != 0:
        raise RuntimeError(
            "No internet access — cannot clone repo.\n"
            "Fix: Notebook options (right panel) → Session options → "
            "Internet → ON, then re-run."
        )
    print("✓ Internet: reachable")

# ── Install runtime dependencies ─────────────────────────────────────────
if IS_KAGGLE:
    _flag = Path("/kaggle/working/.deps_installed")
    _flag.parent.mkdir(parents=True, exist_ok=True)
else:
    _flag = None

if _flag is not None and _flag.exists():
    print("✓ Dependencies already installed (flag found — skipping pip)")
else:
    !pip install -q anthropic pysr pint python-dotenv pyyaml scikit-learn numpy pandas
    # CVE-2024-22190 / CVE-2023-41040 (Dependabot #103/#104) — pin safe GitPython
    !pip install -q "gitpython==3.1.47"
    if _flag is not None:
        _flag.touch()
    print("✓ Dependencies installed")

if src.exists():
    print("✓ Source repo already present at", src.resolve())
else:
    https_url = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
    print(f"Cloning {https_url} → {src} ...")
    !git clone {https_url} {str(src)}
    if _exit_code == 0:
        print("✓ Cloned to", src.resolve())
    else:
        raise RuntimeError(
            f"Clone failed — check output above.\n"
            "If you are behind a proxy, set HTTPS_PROXY and retry.\n"
            "Manual alternative:\n"
            f"  git clone https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git {src}"
        )

# ── Export so cells 0-B and 0-C can pick it up without re-detecting ──────
os.environ["REPO_ROOT_PATH"] = str(src.resolve())
print("REPO_ROOT_PATH:", os.environ["REPO_ROOT_PATH"])


355152b2d573b57663f73422e17a3725db8a95c4	refs/heads/dependabot/pip/gitpython-3.1.47
0f0feea2ded380088a8ae5905b05285c3d25582e	refs/heads/dependabot/pip/python-dotenv-1.2.2
2fe8d065bb3cff6e273087e692e2c352b5b86f36	refs/heads/master
✓ Internet: reachable
✓ Dependencies installed
Cloning https://github.com/sednabcn/LLM-HypatiaX-PAPERS-Public.git → /content/LLM-HypatiaX-PAPERS-Public ...
Cloning into '/content/LLM-HypatiaX-PAPERS-Public'...
remote: Enumerating objects: 944, done.
remote: Counting objects: 100% (298/298), done.
remote: Compressing objects: 100% (203/203), done.
remote: Total 944 (delta 151), reused 196 (delta 75), pack-reused 646 (from 1)
Receiving objects: 100% (944/944), 4.33 MiB | 10.60 MiB/s, done.
Resolving deltas: 100% (496/496), done.
✓ Cloned to /content/LLM-HypatiaX-PAPERS-Public
REPO_ROOT_PATH: /content/LLM-HypatiaX-PAPERS-Public


## 0-B · Verify source tree
> Confirms every file the benchmarks import is present before spending time running.


In [ ]:
# NOTE: experiment_protocol_benchmark.py has been patched to move
# 'from __future__ import annotations' to the top of the file.

import os, sys
from pathlib import Path

# ── Repo root — use the path resolved by the clone cell (0-A) ────────────
# Colab: /content/LLM-HypatiaX-PAPERS-Public
# Kaggle: /kaggle/working/LLM-HypatiaX-PAPERS-Public
# Local: ../LLM-HypatiaX-PAPERS-Public
GITHUB_REPO = "LLM-HypatiaX-PAPERS-Public"
IS_COLAB  = "google.colab" in sys.modules or os.path.exists("/content")
IS_KAGGLE = os.path.exists("/kaggle")
_env_root = os.environ.get("REPO_ROOT_PATH", "")
if _env_root:
    REPO_ROOT = Path(_env_root)
elif IS_COLAB:
    REPO_ROOT = Path("/content") / GITHUB_REPO
elif IS_KAGGLE:
    REPO_ROOT = Path("/kaggle/working") / GITHUB_REPO
else:
    REPO_ROOT = Path(f"../{GITHUB_REPO}")

# hypatiax package: at repo root (public) or nested papers/2025-JMLR/ (private)
HYPATIAX_SRC_PUBLIC  = REPO_ROOT / "hypatiax"
HYPATIAX_SRC_PRIVATE = REPO_ROOT / "papers" / "2025-JMLR" / "hypatiax"

if HYPATIAX_SRC_PUBLIC.exists():
    HYPATIAX_SRC = HYPATIAX_SRC_PUBLIC
    print("Layout : PUBLIC  (hypatiax/ at repo root)")
elif HYPATIAX_SRC_PRIVATE.exists():
    HYPATIAX_SRC = HYPATIAX_SRC_PRIVATE
    print("Layout : PRIVATE (hypatiax/ nested under papers/2025-JMLR/)")
else:
    raise SystemExit(
        f"HypatiaX source not found.\n"
        f"  checked: {HYPATIAX_SRC_PUBLIC.resolve()}\n"
        f"  checked: {HYPATIAX_SRC_PRIVATE.resolve()}\n"
        "Run the clone cell above first."
    )

print(f"HYPATIAX_SRC : {HYPATIAX_SRC.resolve()}\n")

# ── Files checked relative to HYPATIAX_SRC (hypatiax-internal) ───────────
hypatiax_files = [
    # Engine
    "tools/symbolic/hybrid_system_v50_2.py",
    "tools/symbolic/symbolic_engine.py",
    "tools/symbolic/physics_aware_regressor.py",
    "tools/symbolic/smart_structure_detector.py",
    # Validation
    "tools/validation/dimensional_validator.py",
    "tools/validation/domain_validator.py",
    "tools/validation/ensemble_validator.py",
    "tools/validation/symbolic_validator.py",
    # hypatiax/protocols/ library modules
    "protocols/experiment_protocol_defi.py",
    "protocols/experiment_protocol_defi_20.py",
    "protocols/experiment_protocol_nguyen12.py",
    "protocols/experiment_protocol_all_18_a.py",
    "protocols/experiment_protocol_all_20.py",
    "protocols/experiment_protocol_all_30.py",
    "protocols/experiment_protocol_benchmark.py",
    "protocols/experiment_protocol_benchmark_v2.py",
    "protocols/experiment_protocol_comparative.py",
    # Analysis & baselines
    "experiments/comparison/test_suite_comparative_v3.py",
    "experiments/tests/extrapolation_test_protocol.py",
    "analysis/statistical_analysis.py",
    "core/training/baseline_neural_network.py",
    "core/base_pure_llm/baseline_pure_llm.py",
]

# ── Files checked relative to REPO_ROOT (repo-level) ─────────────────────
repo_files = [
    # Root-level protocol runners
    "hypatiax/protocols/experiment_protocol_ablation_exp1.py",
    "hypatiax/protocols/experiment_protocol_defi_v3.py",
    "hypatiax/protocols/experiment_protocol_feynman_exp2.py",
    "hypatiax/protocols/experiment_protocol_nguyen12_exp3.py",
    "hypatiax/protocols/experiment_protocol_noise_sweep.py",
    "hypatiax/protocols/experiment_protocol_hybrid_routing.py",
    "hypatiax/protocols/experiment_protocol_instability_rf02_04.py",
    "hypatiax/protocols/experiment_protocol_extrapolation_comparative.py",
    "hypatiax/protocols/experiment_protocol_provenance_audit.py",
    # Provenance & import-graph tools
    "audit/discover_provenance.py",
    "audit/scan_internal_imports.py",
    # Paper audit notebooks
    "notebooks/NB-01_Citation_Bibliography_Audit.ipynb",
    "notebooks/NB-02_CrossReference_Label_Audit.ipynb",
    "notebooks/NB-03_Section_Structure_Numbering.ipynb",
    "notebooks/NB-04_Numerical_Consistency_Checker.ipynb",
    "notebooks/NB-05_Figure_Image_Dependency_Checker.ipynb",
    "notebooks/NB-06_Code_Quality_Pipeline_Integrity.ipynb",
]
# provenance_map.json is optional (warn, not error)
optional_repo_files = ["provenance_map.json"]

found = 0
total = len(hypatiax_files) + len(repo_files)

print("── hypatiax-internal files (relative to hypatiax/) ─────────")
for f in hypatiax_files:
    ok = (HYPATIAX_SRC / f).exists()
    print(f"  {'✓' if ok else '✗'}  hypatiax/{f}")
    if ok: found += 1

print("\n── repo-root files ──────────────────────────────────────────")
for f in repo_files:
    ok = (REPO_ROOT / f).exists()
    print(f"  {'✓' if ok else '✗'}  {f}")
    if ok: found += 1

print("\n── optional files (warn only) ────────────────────────────────")
for f in optional_repo_files:
    ok = (REPO_ROOT / f).exists()
    print(f"  {'✓' if ok else '⚠ (optional)'}  {f}")

missing = total - found
print(f"\n{found}/{total} critical files present", end="")
if found == total:
    print("  ✓")
else:
    print(f"  ⚠  {missing} missing — check source repo.")


Layout : PUBLIC  (hypatiax/ at repo root)
HYPATIAX_SRC : /content/LLM-HypatiaX-PAPERS-Public/hypatiax

── hypatiax-internal files (relative to hypatiax/) ─────────
  ✓  hypatiax/tools/symbolic/hybrid_system_v50_2.py
  ✓  hypatiax/tools/symbolic/symbolic_engine.py
  ✓  hypatiax/tools/symbolic/physics_aware_regressor.py
  ✓  hypatiax/tools/symbolic/smart_structure_detector.py
  ✓  hypatiax/tools/validation/dimensional_validator.py
  ✓  hypatiax/tools/validation/domain_validator.py
  ✓  hypatiax/tools/validation/ensemble_validator.py
  ✓  hypatiax/tools/validation/symbolic_validator.py
  ✓  hypatiax/protocols/experiment_protocol_defi.py
  ✓  hypatiax/protocols/experiment_protocol_defi_20.py
  ✓  hypatiax/protocols/experiment_protocol_nguyen12.py
  ✓  hypatiax/protocols/experiment_protocol_all_18_a.py
  ✓  hypatiax/protocols/experiment_protocol_all_20.py
  ✓  hypatiax/protocols/experiment_protocol_all_30.py
  ✓  hypatiax/protocols/experiment_protocol_benchmark.py
  ✓  hypatiax/protocols/ex

## 0-C · Set `sys.path` and environment variables
> Run once. All subsequent cells inherit these settings.


In [ ]:
import sys, os
from pathlib import Path

# ── Repo root — platform-aware (mirrors 0-A clone cell) ──────────────────
GITHUB_REPO = "LLM-HypatiaX-PAPERS-Public"
IS_COLAB  = "google.colab" in sys.modules or os.path.exists("/content")
IS_KAGGLE = os.path.exists("/kaggle")
_env_root = os.environ.get("REPO_ROOT_PATH", "")
if _env_root:
    REPO_BASE = Path(_env_root)
elif IS_COLAB:
    REPO_BASE = Path("/content") / GITHUB_REPO
elif IS_KAGGLE:
    REPO_BASE = Path("/kaggle/working") / GITHUB_REPO
else:
    REPO_BASE = Path(f"../{GITHUB_REPO}")
REPRO_ROOT = str(REPO_BASE.resolve())

# ── hypatiax package location (public: repo root / private: papers/2025-JMLR/) ─
HYPATIAX_SRC_PUBLIC  = REPO_BASE / "hypatiax"
HYPATIAX_SRC_PRIVATE = REPO_BASE / "papers" / "2025-JMLR" / "hypatiax"

if HYPATIAX_SRC_PUBLIC.exists():
    HYPATIAX_SRC       = str(HYPATIAX_SRC_PUBLIC.resolve())
    HYPATIAX_IMPORT_ROOT = str(REPO_BASE.resolve())
elif HYPATIAX_SRC_PRIVATE.exists():
    HYPATIAX_SRC         = str(HYPATIAX_SRC_PRIVATE.resolve())
    HYPATIAX_IMPORT_ROOT = str((REPO_BASE / "papers" / "2025-JMLR").resolve())
else:
    raise SystemExit(
        f"HypatiaX source not found.\n"
        f"  checked: {HYPATIAX_SRC_PUBLIC.resolve()}\n"
        f"  checked: {HYPATIAX_SRC_PRIVATE.resolve()}\n"
        "Run the clone cell (0-A) first."
    )

# ── sys.path ──────────────────────────────────────────────────────────────
for p in [HYPATIAX_IMPORT_ROOT, REPRO_ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)

# ── environment variables ─────────────────────────────────────────────────
os.environ["REPRO_ROOT"]           = REPRO_ROOT
os.environ["HYPATIAX_ROOT"]        = HYPATIAX_SRC
os.environ["HYPATIAX_IMPORT_ROOT"] = HYPATIAX_IMPORT_ROOT
os.environ["PYTHONPATH"] = os.pathsep.join(
    [HYPATIAX_IMPORT_ROOT, REPRO_ROOT]
    + [p for p in os.environ.get("PYTHONPATH", "").split(os.pathsep) if p]
)
os.environ["PYTHON_JULIACALL_HANDLE_SIGNALS"] = "yes"

# ── Change CWD to repo root so all subsequent !python calls resolve ───────
# (Jupyter !python runs from the kernel's CWD, not the notebook location)
os.chdir(REPRO_ROOT)
print("CWD changed to:", os.getcwd())

print("REPRO_ROOT          :", REPRO_ROOT)
print("HYPATIAX_SRC        :", HYPATIAX_SRC)
print("HYPATIAX_IMPORT_ROOT:", HYPATIAX_IMPORT_ROOT)

harness = [
    "hypatiax/protocols/_base.py", "hypatiax/protocols/universal_protocol.py",
    "hypatiax/core/runners/common.py",
    "scripts/patches/apply_patches.py", "hypatiax/reproducibility/hash_lock.py",
    "config/repro.yaml",
]
# Verify config/repro.yaml has the correct model string
import yaml as _yaml
repro_cfg = Path(REPRO_ROOT, "config", "repro.yaml")
if repro_cfg.exists():
    try:
        cfg = _yaml.safe_load(repro_cfg.read_text())
        model = cfg.get("llm_model", "")
        _VALID_MODELS = {"claude-sonnet-4-20250514"}
        if model not in _VALID_MODELS:
            print(f"  ⚠  config/repro.yaml llm_model={model!r} — expected one of {_VALID_MODELS}")
            print("     Fix: set llm_model: claude-sonnet-4-20250514  in config/repro.yaml")
        else:
            print(f"  ✓  config/repro.yaml llm_model={model!r}")
    except Exception:
        print("  ⚠  could not parse config/repro.yaml")
print("\nRepro harness:")
for f in harness:
    print(f"  {'✓' if Path(REPRO_ROOT, f).exists() else '✗'}  {f}")

# ── v7: fixup-init (FIX-INIT-PY) — guard broken HypatiaX import ──────────
# Mirrors run_all_checkpoint.py fixup-init step.  Wraps the bare
# `from hypatiax.core import HypatiaX` line in hypatiax/__init__.py inside
# a try/except so sub-packages (hypatiax.protocols.*, etc.) remain
# importable even when hypatiax.core is broken or not compiled.
import pathlib as _pl_fi
_fi_init = _pl_fi.Path(REPRO_ROOT) / 'hypatiax' / '__init__.py'
if _fi_init.exists():
    _fi_src = _fi_init.read_text(encoding='utf-8')
    _BAD  = 'from hypatiax.core import HypatiaX'
    _GOOD = ('try:\n'
             '    from hypatiax.core import HypatiaX  # noqa: F401\n'
             'except Exception:  # broken core does not block sub-packages\n'
             '    HypatiaX = None  # type: ignore')
    if _BAD in _fi_src and 'except Exception:' not in _fi_src:
        _fi_init.write_text(_fi_src.replace(_BAD, _GOOD), encoding='utf-8')
        print('  ✓ fixup-init: hypatiax/__init__.py patched — HypatiaX import guarded')
    else:
        print('  ✓ fixup-init: already patched or BAD string absent')
else:
    print('  ⚠ fixup-init: hypatiax/__init__.py not found — skipping')


CWD changed to: /content/LLM-HypatiaX-PAPERS-Public
REPRO_ROOT          : /content/LLM-HypatiaX-PAPERS-Public
HYPATIAX_SRC        : /content/LLM-HypatiaX-PAPERS-Public/hypatiax
HYPATIAX_IMPORT_ROOT: /content/LLM-HypatiaX-PAPERS-Public
  ✓  config/repro.yaml llm_model='claude-sonnet-4-20250514'

Repro harness:
  ✓  hypatiax/protocols/_base.py
  ✓  hypatiax/protocols/universal_protocol.py
  ✓  hypatiax/core/runners/common.py
  ✓  scripts/patches/apply_patches.py
  ✓  hypatiax/reproducibility/hash_lock.py
  ✓  config/repro.yaml
  ✓ fixup-init: already patched or BAD string absent


## 1 · API key (Anthropic Claude)


In [ ]:
import os
from pathlib import Path

# Option 1 — set directly (do not commit with a real key)
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

# Option 2 — .env file at repo root (recommended for local runs)
try:
    from dotenv import load_dotenv
    load_dotenv(Path(".env"))
    print(".env loaded")
except ImportError:
    pass

# Option 3 — Google Colab secrets
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("Colab userdata loaded")
except Exception:
    pass

# Option 4 — Kaggle UserSecrets (Kaggle notebook environment)
# UserSecretsClient.get_secret() returns None (not an exception) when the
# secret is absent — guard explicitly so we don't silently store "None".
try:
    from kaggle_secrets import UserSecretsClient
    _kag_key = UserSecretsClient().get_secret("ANTHROPIC_API_KEY")
    if _kag_key:
        os.environ["ANTHROPIC_API_KEY"] = _kag_key
        print("Kaggle UserSecrets loaded")
    else:
        print("⚠  Kaggle UserSecrets: ANTHROPIC_API_KEY not found.")
        print("   Add it via: Add-ons → Secrets → + Add secret → ANTHROPIC_API_KEY")
except Exception:
    pass

key = os.environ.get("ANTHROPIC_API_KEY", "")
if not key:
    raise RuntimeError(
        "ANTHROPIC_API_KEY is not set.\n"
        "Colab: Secrets panel → add ANTHROPIC_API_KEY.\n"
        "Kaggle: Add-ons → Secrets → add ANTHROPIC_API_KEY.\n"
        "Local: export ANTHROPIC_API_KEY='sk-ant-...' or add to .env"
    )
print(f"API key set: YES ({len(key)} chars)")


.env loaded
Colab userdata loaded
API key set: YES (108 chars)


## 2 · Runtime configuration
Mirrors `config/repro.yaml`. Set once; all `!python` calls below inherit via environment.


In [ ]:
# ── Runtime config — PAPER-QUALITY (FAST=0 hardcoded) ──────────────────
# Synced with run_all_checkpoint.py v4.8 — all env var names match exactly
# For smoke-tests use HypatiaX_Experiments_v7_PUBLIC_fast.ipynb
import os

FAST = 0  # hardcoded — do NOT change
os.environ['FAST'] = '0'

# Seeds & model
os.environ.setdefault('NN_SEED',    '42')
os.environ.setdefault('PYSR_SEED',  '42')
os.environ.setdefault('PYTHONHASHSEED', os.environ.get('PYSR_SEED', '42'))
os.environ.setdefault('LLM_MODEL', 'claude-sonnet-4-20250514')
os.environ['LLM_RETRIES']  = '3'
os.environ['LLM_K_RUNS']   = '1'  # NOTE: instability cell overrides to K=30 locally (§10.9)
os.environ['ENGINE_NAME']  = 'hybrid_system_v50_2'

# Task counts — FULL paper values
os.environ['N_TASKS_DEFI']        = '74'
os.environ['N_TASKS_INSTABILITY'] = '70'

# Neural network
os.environ['PCA_TRAIN_FRAC']  = '0.40'
os.environ['NN_TIME_LIMIT']   = '120'

# PySR — paper-quality values
# run_all_checkpoint.py reads: N_ITERATIONS, POPULATIONS, PYSR_TIMEOUT, METHOD_TIMEOUT
# Both canonical and legacy names set so all scripts see consistent values
os.environ['N_ITERATIONS']          = '1000'   # run_all_checkpoint.py canonical name
os.environ['POPULATIONS']           = '30'     # run_all_checkpoint.py canonical name
os.environ['PYSR_NITERATIONS']      = '1000'   # legacy alias (some scripts read this)
os.environ['PYSR_POPULATIONS']      = '30'     # legacy alias (some scripts read this)
os.environ['PYSR_TIMEOUT']          = '1100'   # seconds per equation attempt
os.environ['METHOD_TIMEOUT']        = '900'    # seconds per full method call
os.environ['PYSR_PARALLELISM']      = 'multithreading'
os.environ['PYSR_POPULATION_SIZE']  = '33'
os.environ['PYSR_PARSIMONY']        = '0.01'
os.environ['PYSR_MAXSIZE']          = '30'

# v8: K=30 is injected locally in the instability cell via its own env dict.
# LLM_K_RUNS=1 here is the default for all other cells.
# N_TASKS_INSTABILITY=70 (70 tasks × K=30) matches paper §10.9.

# ONE_EQUATION must be UNSET for full runs
os.environ.pop('ONE_EQUATION', None)

# Feynman / Nguyen / noise — FULL counts
os.environ['N_FEYNMAN_TASKS']   = '30'
os.environ['N_NGUYEN_TASKS']    = '12'

# DeFi benchmark module-level constants
os.environ['_METHOD_TIMEOUT_SECS'] = '900'
os.environ['_PYSR_TIMEOUT_SECS']   = '1100'

# v7: exp1b env vars
os.environ['DEFI_V3C_NO_TIMEOUT_FLAGS'] = '1'
os.environ['DEFI_TASK_FILTER']          = 'portfolio'
os.environ['DEFI_SEEDS']               = '42,99,123,777,2024'

# v7: exp3/exp3b env var
os.environ['SKIP_PKG_CHECK'] = '1'

# v7: suppA / extrap env vars
os.environ['SKIP_PERF_ANALYSIS']    = '1'
os.environ['HYPATIAX_CORE_OPTIONAL'] = '1'

# v7: verify env vars (FIX-VERIFY)
import pathlib as _pl
_repro = _pl.Path(os.environ.get('REPRO_ROOT', '.'))
os.environ['PATCHED_DATA_DIR']   = str(_repro / 'hypatiax' / 'data' / 'results')
os.environ['VERIFY_RESULTS_DIR'] = str(_repro / 'hypatiax' / 'data' / 'results')

# v7: tables env vars (FIX-TABLES)
os.environ['TABLE_OUTDIR'] = str(_repro / 'hypatiax' / 'data' / 'results' / 'tables')

print('Config — FULL (paper-quality, FAST=0)')
print(f'  PySR  : {os.environ["N_ITERATIONS"]} iters · {os.environ["POPULATIONS"]} pops · {os.environ["PYSR_TIMEOUT"]}s/eq · {os.environ["METHOD_TIMEOUT"]}s/method')
print(f'  Tasks : DeFi={os.environ["N_TASKS_DEFI"]} · Feynman={os.environ["N_FEYNMAN_TASKS"]} · Nguyen={os.environ["N_NGUYEN_TASKS"]} · Instability={os.environ["N_TASKS_INSTABILITY"]}')
print(f'  NN    : {os.environ["NN_TIME_LIMIT"]}s limit · LLM retries={os.environ["LLM_RETRIES"]}')
print(f'  Model : {os.environ["LLM_MODEL"]}')
print(f'  PySR parallelism: {os.environ["PYSR_PARALLELISM"]}')
print('  Paper-quality mode — results WILL match paper targets.')

# Ensure output subdirectory tree exists
import pathlib
_results = pathlib.Path(os.environ.get('REPRO_ROOT', '.')) / 'hypatiax' / 'data' / 'results'
for _sub in [
    'comparison_results/extrapolation', 'comparison_results/feynman-tests/noise-sweep',
    'comparison_results/noise-noiseless/noiseless', 'comparison_results/noise-noiseless/15',
    'extrapolation', 'hybrid_llm_nn/all_domains', 'hybrid_llm_nn/defi',
    'hybrid_pysr/all_domains', 'hybrid_pysr/defi', 'llm_guided/all_domains',
    'llm_guided/defi', 'standalone_llm_nn', 'figures', 'tables',
]:
    (_results / _sub).mkdir(parents=True, exist_ok=True)
print('Output subdirs ready:', _results)


Config — FULL (paper-quality, FAST=0)
  PySR  : 1000 iters · 30 pops · 1100s/eq · 900s/method
  Tasks : DeFi=74 · Feynman=30 · Nguyen=12 · Instability=70
  NN    : 120s limit · LLM retries=3
  Model : claude-sonnet-4-20250514
  PySR parallelism: multithreading
  Paper-quality mode — results WILL match paper targets.
Output subdirs ready: /content/LLM-HypatiaX-PAPERS-Public/hypatiax/data/results


## Kaggle: Checkpoint / Resume Helper
> **Kaggle only** — tracks finished experiments so you can resume after the 12-h session restarts.
> Safe to skip on Colab Pro+.


In [ ]:
import os, json, pathlib, datetime

# ── Checkpoint file location ──────────────────────────────────────────────
# Kaggle: /kaggle/working/ persists across sessions when Output is saved.
# Colab:  repo root (REPRO_ROOT) — persists if Drive is mounted.
# Local:  repo root.
def _cp_path():
    if os.path.exists('/kaggle'):
        return pathlib.Path('/kaggle/working/hypatiax_checkpoint.json')
    repro = os.environ.get('REPRO_ROOT', '')
    if repro:
        return pathlib.Path(repro) / 'hypatiax_checkpoint.json'
    return pathlib.Path('hypatiax_checkpoint.json')

CHECKPOINT_FILE = _cp_path()

def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        try:
            return json.loads(CHECKPOINT_FILE.read_text())
        except Exception:
            pass
    return {'completed': [], 'started': {}}

def save_checkpoint(exp_name):
    cp = load_checkpoint()
    if exp_name not in cp['completed']:
        cp['completed'].append(exp_name)
    cp.setdefault('started', {}).pop(exp_name, None)
    cp['last_updated'] = datetime.datetime.now().isoformat()
    CHECKPOINT_FILE.parent.mkdir(parents=True, exist_ok=True)
    CHECKPOINT_FILE.write_text(json.dumps(cp, indent=2))
    print(f'  ✓ Checkpoint saved: {exp_name}')

def mark_started(exp_name):
    cp = load_checkpoint()
    cp.setdefault('started', {})[exp_name] = datetime.datetime.now().isoformat()
    CHECKPOINT_FILE.parent.mkdir(parents=True, exist_ok=True)
    CHECKPOINT_FILE.write_text(json.dumps(cp, indent=2))

def is_done(exp_name):
    return exp_name in load_checkpoint().get('completed', [])

def reset_checkpoint(exp_name=None):
    """Remove one experiment (or all) from completed so it re-runs."""
    cp = load_checkpoint()
    if exp_name:
        cp['completed'] = [e for e in cp['completed'] if e != exp_name]
        cp.setdefault('started', {}).pop(exp_name, None)
        print(f'  Reset: {exp_name} will re-run next session')
    else:
        cp = {'completed': [], 'started': {}}
        print('  Reset: all experiments will re-run')
    CHECKPOINT_FILE.write_text(json.dumps(cp, indent=2))

# ── Session summary ───────────────────────────────────────────────────────
ALL_EXPS = ['exp1', 'exp1b', 'exp2', 'exp3', 'exp3b', 'suppB', 'suppA',
            'instability', 'extrap', 'provenance']

cp = load_checkpoint()
done = cp.get('completed', [])
todo = [e for e in ALL_EXPS if e not in done]

print('Checkpoint :', CHECKPOINT_FILE)
print('Completed  :', done if done else '(none yet)')
print('Remaining  :', todo if todo else '— ALL DONE')
if cp.get('last_updated'):
    print('Last update:', cp['last_updated'])
print()
if todo:
    print('Re-run this notebook from the top each new Kaggle session.')
    print('Completed experiments are skipped automatically.')
else:
    print('All experiments complete — proceed to validation (cell 14).')


Checkpoint : /kaggle/working/hypatiax_checkpoint.json
Completed  : (none yet)
Remaining  : ['exp1', 'exp1b', 'exp2', 'exp3', 'exp3b', 'suppB', 'suppA', 'instability', 'extrap', 'provenance']

Re-run this notebook from the top each new Kaggle session.
Completed experiments are skipped automatically.


## 💾 Auto-backup Helper
> `save_results(exp_name)` backs up results + checkpoint to Drive/Kaggle persistent storage.  
> Call it at the end of each experiment cell so a crash only loses the current run.


In [ ]:
# ── Auto-backup results to persistent Drive/Kaggle storage ───────────
# Call save_results(exp_name) after each experiment finishes.
# On Colab: copies results + checkpoint to /content/drive/MyDrive/HypatiaX/
# On Kaggle: copies to /kaggle/working/HypatiaX_persistent/
# This means after a crash you only lose the current in-progress experiment.
import os, shutil, pathlib, datetime, json

def save_results(exp_name=None):
    """Back up results + checkpoint to persistent storage after each experiment."""
    repro  = pathlib.Path(os.environ.get('REPRO_ROOT', '.')).resolve()
    pers   = pathlib.Path(os.environ.get('PERSISTENT_DIR', str(repro))).resolve()
    
    results_src = repro / 'hypatiax' / 'data' / 'results'
    logs_src    = repro / 'logs'
    cp_src      = logs_src / 'pipeline_checkpoint.json'
    nb_cp_src   = repro / 'hypatiax_checkpoint.json'
    cp_pers     = pers / 'pipeline_checkpoint.json'
    
    # Back up checkpoint files
    for src in [cp_src, nb_cp_src]:
        if src.exists():
            shutil.copy2(src, cp_pers)
            break
    
    # Back up results directory
    if results_src.exists():
        results_dst = pers / 'results'
        shutil.copytree(results_src, results_dst, dirs_exist_ok=True)
    
    # Back up logs
    if logs_src.exists():
        logs_dst = pers / 'logs'
        shutil.copytree(logs_src, logs_dst, dirs_exist_ok=True)
    
    ts = datetime.datetime.now().strftime('%H:%M:%S')
    msg = f"[{ts}] Backed up"
    if exp_name:
        msg += f" after {exp_name}"
        # Also update notebook checkpoint
        save_checkpoint(exp_name)
    print(f'✅ {msg} → {pers}')

print('✅ save_results() helper ready')
print(f'   Persistent storage: {os.environ.get("PERSISTENT_DIR", "(not set — run Restore Session cell first)")}')


## 3 · Apply patches
Must run **before any benchmark**. Applies:

**Step 3a** — `apply_patches.py` (FIX-C1 · FIX-C2 · FIX-T1/T2/XR3)
- **FIX-C1** — rename duplicate DeFi case names
- **FIX-C2** — replace any stray `hybrid_system_v40` references with `hybrid_system_v50_2`
- **FIX-T1/T2/XR3** — paper text corrections

**Step 3b** — inline fixes (run before import verification)
- **FIX-METRICS** — create `hypatiax/core/metrics.py` (`compute_r2` etc.) so `statistical_analysis` imports cleanly
- **FIX-FUTURE** — move `from __future__ import annotations` to line 1 in `experiment_protocol_benchmark.py`
- **FIX-LOCKS** — patch `run_all_checkpoint.py` `_clear_stale_locks()` so it never rglobs from `/` (Colab `/proc` crash)

**Step 3c** — verify all imports resolve (cell immediately below)


In [ ]:
import os, sys, subprocess
os.chdir(os.environ.get('REPRO_ROOT', '.'))
repro = os.environ.get('REPRO_ROOT', '.')

for script in [
    'scripts/patches/generate_patches.py',
    'scripts/patches/apply_patches.py',
    ['scripts/patches/apply_patches.py', '--verify'],
]:
    cmd = [sys.executable] + ([script] if isinstance(script, str) else script)
    proc = subprocess.Popen(cmd, cwd=repro, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f'Script failed (exit {proc.returncode}): {cmd}')
print('\n✅ Patches applied and verified')


In [ ]:
import os, pathlib, re

_repro_root = pathlib.Path(os.environ.get("REPRO_ROOT", "."))

HYPATIAX_SRC_PUBLIC  = _repro_root / "hypatiax"
HYPATIAX_SRC_PRIVATE = _repro_root / "papers" / "2025-JMLR" / "hypatiax"

if HYPATIAX_SRC_PUBLIC.exists():
    HYPATIAX_SRC = HYPATIAX_SRC_PUBLIC.resolve()
elif HYPATIAX_SRC_PRIVATE.exists():
    HYPATIAX_SRC = HYPATIAX_SRC_PRIVATE.resolve()
else:
    raise RuntimeError("HYPATIAX_SRC could not be determined. Run setup cells (0-A, 0-C) first.")

# ── FIX-FUTURE: experiment_protocol_benchmark.py ────────────────────────────
_bench = HYPATIAX_SRC / "protocols" / "experiment_protocol_benchmark.py"
if _bench.exists():
    _content = _bench.read_text()
    _future  = [l.strip() for l in _content.splitlines() if l.strip().startswith("from __future__")]
    _other   = [l for l in _content.splitlines() if not l.strip().startswith("from __future__")]
    _new     = "\n".join(sorted(set(_future)) + ["\n"] + _other)
    _new     = re.sub(r'\n{3,}', '\n\n', _new)
    if _new != _content:
        _bench.write_text(_new)
        print(f"✓ Fixed from __future__ imports in {_bench}")
    else:
        print(f"✓ from __future__ already correct in {_bench.name}")
else:
    print(f"⚠ {_bench} not found — skipping")

# ── FIX-METRICS: create hypatiax/core/metrics.py ────────────────────────────
_metrics_path = HYPATIAX_SRC / "core" / "metrics.py"
_metrics_path.parent.mkdir(parents=True, exist_ok=True)

METRICS_SOURCE = '''
"""
hypatiax/core/metrics.py — regression & SR evaluation metrics for HypatiaX.
Auto-generated by HypatiaX_Experiments_v10.ipynb (Fix #11).
"""
from __future__ import annotations
import math, warnings
from typing import Sequence, Union
import numpy as np

ArrayLike = Union[Sequence[float], np.ndarray]

def _to_array(x, name="array"):
    arr = np.asarray(x, dtype=np.float64).ravel()
    if len(arr) == 0:
        raise ValueError(f"{name} must not be empty")
    return arr

def _validate_pair(y_true, y_pred):
    yt, yp = _to_array(y_true, "y_true"), _to_array(y_pred, "y_pred")
    if len(yt) != len(yp):
        raise ValueError(f"Length mismatch: {len(yt)} vs {len(yp)}")
    return yt, yp

def compute_r2(y_true, y_pred):
    """R² (coefficient of determination). Returns 0.0 for constant targets."""
    yt, yp = _validate_pair(y_true, y_pred)
    ss_res = float(np.sum((yt - yp) ** 2))
    ss_tot = float(np.sum((yt - np.mean(yt)) ** 2))
    if ss_tot == 0.0:
        return 0.0 if ss_res != 0.0 else 1.0
    return float(1.0 - ss_res / ss_tot)

def compute_mse(y_true, y_pred):
    """Mean Squared Error."""
    yt, yp = _validate_pair(y_true, y_pred)
    return float(np.mean((yt - yp) ** 2))

def compute_rmse(y_true, y_pred):
    """Root Mean Squared Error."""
    return math.sqrt(compute_mse(y_true, y_pred))

def compute_mae(y_true, y_pred):
    """Mean Absolute Error."""
    yt, yp = _validate_pair(y_true, y_pred)
    return float(np.mean(np.abs(yt - yp)))

def compute_max_error(y_true, y_pred):
    """Maximum absolute error."""
    yt, yp = _validate_pair(y_true, y_pred)
    return float(np.max(np.abs(yt - yp)))

def compute_near_perfect_rate(scores, threshold=0.99):
    """Fraction of R² scores above threshold (paper target §10.2: 89.2% at 0.99)."""
    arr = _to_array(scores, "scores")
    return float(np.mean(arr > threshold))

def compute_catastrophic_rate(scores, threshold=-1.0):
    """Fraction of R² scores below threshold (paper target §10.2: 0 failures)."""
    arr = _to_array(scores, "scores")
    return float(np.mean(arr < threshold))

def compute_speedup(t_baseline, t_system):
    """Speedup ratio t_baseline/t_system (paper target §10.4: 1.73×)."""
    if t_system == 0.0:
        return float("inf")
    return float(t_baseline / t_system)

def evaluate_all(y_true, y_pred, *, near_perfect_threshold=0.99, catastrophic_threshold=-1.0):
    """Return dict with r2, mse, rmse, mae, max_error, near_perfect, catastrophic."""
    r2 = compute_r2(y_true, y_pred)
    return {
        "r2":           r2,
        "mse":          compute_mse(y_true, y_pred),
        "rmse":         compute_rmse(y_true, y_pred),
        "mae":          compute_mae(y_true, y_pred),
        "max_error":    compute_max_error(y_true, y_pred),
        "near_perfect": bool(r2 > near_perfect_threshold),
        "catastrophic": bool(r2 < catastrophic_threshold),
    }

# Backwards-compat aliases (sklearn-style names)
r2_score = compute_r2
mean_squared_error      = compute_mse
root_mean_squared_error = compute_rmse
mean_absolute_error     = compute_mae
'''

if not _metrics_path.exists():
    _metrics_path.write_text(METRICS_SOURCE.lstrip())
    print(f"✓ Created {_metrics_path}")
else:
    # Check it has compute_r2; if not (placeholder), overwrite
    _existing = _metrics_path.read_text()
    if "compute_r2" not in _existing:
        _metrics_path.write_text(METRICS_SOURCE.lstrip())
        print(f"✓ Replaced placeholder {_metrics_path}")
    else:
        print(f"✓ {_metrics_path.name} already has compute_r2 — left unchanged")

# ── FIX-SA: verify statistical_analysis.py import ───────────────────────────
# statistical_analysis.py uses:
#   from hypatiax.core.metrics import compute_r2   (line 85)
# This works automatically once hypatiax/core/metrics.py (above) is created.
# The only thing to fix is if a previous patch incorrectly redirected it
# to `from hypatiax.core.runners import common as metrics`.
_sa = HYPATIAX_SRC / "analysis" / "statistical_analysis.py"
if _sa.exists():
    _sa_content = _sa.read_text()
    _bad_redirect = "from hypatiax.core.runners import common as metrics"
    _correct_import = "from hypatiax.core.metrics import compute_r2"
    if _bad_redirect in _sa_content:
        # Undo the wrong redirect; restore the original direct import
        _sa.write_text(_sa_content.replace(_bad_redirect, _correct_import))
        print(f"✓ Reverted bad redirect → restored '{_correct_import}' in {_sa.name}")
    elif _correct_import in _sa_content:
        print(f"✓ {_sa.name} already has the correct import — no changes needed")
    else:
        print(f"⚠ Unexpected import pattern in {_sa.name} — inspect manually")
else:
    print(f"⚠ {_sa} not found — skipping")


✓ Fixed from __future__ imports in /content/LLM-HypatiaX-PAPERS-Public/hypatiax/protocols/experiment_protocol_benchmark.py
✓ metrics.py already has compute_r2 — left unchanged
✓ statistical_analysis.py already has the correct import — no changes needed


In [ ]:
# ── FIX-LOCKS: patch run_all_checkpoint.py — _clear_stale_locks OSError ─────
# Root cause: Path(sys.executable).parent.parent.parent resolves to / on Colab
# (/usr/bin/python3.x → /usr/bin → /usr → /), then rglob walks /proc and
# crashes with OSError [Errno 22] Invalid argument on pseudo-filesystem entries.
# Fix: remove the / -reaching path; add a blocklist; wrap both rglob calls.

from pathlib import Path

_rac = Path(os.environ.get("REPRO_ROOT", ".")) / "run_all_checkpoint.py"
if not _rac.exists():
    # fallback: same directory as this notebook
    import pathlib as _pl
    _rac = _pl.Path(".").resolve() / "run_all_checkpoint.py"

if not _rac.exists():
    print(f"⚠ run_all_checkpoint.py not found at {_rac} — skipping patch")
else:
    _src = _rac.read_text()

    OLD_LOCKS = """    # ── 2. Julia / juliapkg lock.pid ──────────────────────────────────────
    # Search: active venv, parent dirs, and common local Python install paths.
    _julia_roots = [
        Path(_sys_locks.executable).parent.parent,
        Path(_sys_locks.executable).parent.parent.parent,
        Path.home() / ".local",
        Path.home() / "Downloads" / "py312",
        Path.home() / "Downloads" / "py311",
        Path.home() / "Downloads" / "py310",
    ]
    for _root in _julia_roots:
        if _root.exists():
            for _pid in _root.rglob("julia_env/lock.pid"):
                _try_unlink(_pid)"""

    NEW_LOCKS = """    # ── 2. Julia / juliapkg lock.pid ──────────────────────────────────────
    # .parent.parent.parent excluded: on Colab sys.executable is
    # /usr/bin/python3.x so three .parent calls reach / → rglob walks /proc.
    _exe = Path(_sys_locks.executable).resolve()
    _julia_roots = [
        _exe.parent.parent,
        Path.home() / ".local",
        Path.home() / ".julia" / "environments",
        Path.home() / "Downloads" / "py312",
        Path.home() / "Downloads" / "py311",
        Path.home() / "Downloads" / "py310",
    ]
    _BLOCKED = {Path("/"), Path("/usr"), Path("/usr/local")}
    for _root in _julia_roots:
        if not _root.exists() or _root in _BLOCKED:
            continue
        try:
            for _pid in _root.rglob("julia_env/lock.pid"):
                _try_unlink(_pid)
        except OSError:
            pass"""

    OLD_REPO = """    # ── 4. Any lock.pid under repo root ───────────────────────────────────
    for lf in REPO_ROOT.rglob("lock.pid"):
        _try_unlink(lf)"""

    NEW_REPO = """    # ── 4. Any lock.pid under repo root ───────────────────────────────────
    try:
        for lf in REPO_ROOT.rglob("lock.pid"):
            _try_unlink(lf)
    except OSError:
        pass"""

    _changed = False
    if OLD_LOCKS in _src:
        _src = _src.replace(OLD_LOCKS, NEW_LOCKS, 1)
        _src = _src.replace(OLD_REPO,  NEW_REPO,  1)
        _rac.write_text(_src)
        _changed = True
        print(f"✓ Patched _clear_stale_locks in {_rac.name}")
    else:
        print(f"✓ {_rac.name} already patched or pattern changed — skipping")

    if _changed:
        import ast
        try:
            ast.parse(_rac.read_text())
            print("✓ Syntax check passed")
        except SyntaxError as e:
            print(f"✗ Syntax error after patch: {e}")


✓ run_all_checkpoint.py already patched or pattern changed — skipping


In [ ]:
# ── FIX-WALLCLOCK: patch exp1 script wall-clock formula ─────────────────────
# The ablation script computes: wall_clock = timeout_s * 2.1
# where timeout_s = min(METHOD_TIMEOUT, 300) = 300.
# So wall_clock = 630s even though PYSR_TIMEOUT=1100s.
# Fix: replace the formula so it respects PYSR_TIMEOUT.
import pathlib, os, re

_repo = pathlib.Path(os.environ.get('REPRO_ROOT', '.'))
_script = _repo / 'hypatiax' / 'experiments' / 'benchmarks' / 'exp1_ablation_populations_30_updated.py'

if _script.exists():
    txt = _script.read_text()
    # Match patterns like: N * 2.1, seconds * 2.1, timeout_s * 2.1
    _old = re.search(r'(wall.clock.{0,20}?)(\w+)\s*\*\s*2\.1', txt)
    if _old:
        # Replace multiplier expression with max(value, PYSR_TIMEOUT) * 1.15
        old_expr = _old.group(0)
        var = _old.group(2)
        new_expr = f'max({var}, int(os.environ.get("PYSR_TIMEOUT", 1100))) * 1.15'
        patched = txt.replace(old_expr, _old.group(1) + new_expr, 1)
        # Ensure 'import os' is at the top if not present
        if 'import os' not in patched[:200]:
            patched = 'import os\n' + patched
        _script.write_text(patched)
        print(f'✓ FIX-WALLCLOCK applied: {old_expr!r} → {new_expr!r}')
    else:
        print('⚠ FIX-WALLCLOCK: pattern not found — searching for alternative...')
        # Fallback: look for WallClockTimeout or signal handler setup
        if 'WallClockTimeout' in txt or 'wall_clock' in txt.lower():
            print('  Wall-clock code found but pattern differs — review manually')
        else:
            print('  No wall-clock code found — may already be fixed or uses different mechanism')
else:
    print(f'⚠ Script not found at {_script} — run after repo clone')


⚠ FIX-WALLCLOCK: pattern not found — searching for alternative...
  Wall-clock code found but pattern differs — review manually


## 4 · Verify all imports resolve


In [ ]:
# NOTE: experiment_protocol_benchmark.py has been patched to move
# 'from __future__ import annotations' to the top of the file.

import importlib, sys

importlib.invalidate_caches()

# Clear stale cached modules so fixes take effect
for m in [k for k in sys.modules if k.startswith(("hypatiax", "protocols", "core", "shared"))]:
    del sys.modules[m]

checks = {
    # Repro harness
    "protocols._base"                                  : "hypatiax.protocols._base",
    "protocols.universal_protocol"                     : "hypatiax.protocols.universal_protocol",
    "core.runners.common"                              : "hypatiax.core.runners.common",
    # Engine
    "hybrid_system_v50_2"                              : "hypatiax.tools.symbolic.hybrid_system_v50_2",
    "symbolic_engine"                                  : "hypatiax.tools.symbolic.symbolic_engine",
    "physics_aware_regressor"                          : "hypatiax.tools.symbolic.physics_aware_regressor",
    "smart_structure_detector"                         : "hypatiax.tools.symbolic.smart_structure_detector",
    # Validation
    "dimensional_validator"                            : "hypatiax.tools.validation.dimensional_validator",
    "domain_validator"                                 : "hypatiax.tools.validation.domain_validator",
    "ensemble_validator"                               : "hypatiax.tools.validation.ensemble_validator",
    "symbolic_validator"                               : "hypatiax.tools.validation.symbolic_validator",
    # Protocol scripts (authoritative runners)
    "experiment_protocol_ablation_exp1"                : "hypatiax.protocols.experiment_protocol_ablation_exp1",
    "experiment_protocol_defi_v3"                      : "hypatiax.protocols.experiment_protocol_defi_v3",
    "experiment_protocol_feynman_exp2"                 : "hypatiax.protocols.experiment_protocol_feynman_exp2",
    "experiment_protocol_nguyen12_exp3"                : "hypatiax.protocols.experiment_protocol_nguyen12_exp3",
    "experiment_protocol_noise_sweep"                  : "hypatiax.protocols.experiment_protocol_noise_sweep",
    "experiment_protocol_hybrid_routing"               : "hypatiax.protocols.experiment_protocol_hybrid_routing",
    "experiment_protocol_instability_rf02_04"          : "hypatiax.protocols.experiment_protocol_instability_rf02_04",
    "experiment_protocol_extrapolation_comparative"    : "hypatiax.protocols.experiment_protocol_extrapolation_comparative",
    "experiment_protocol_provenance_audit"             : "hypatiax.protocols.experiment_protocol_provenance_audit",
    # hypatiax/protocols/ — input-data/library modules (imported by root protocols/)
    "experiment_protocol_defi"           : "hypatiax.protocols.experiment_protocol_defi",
    "experiment_protocol_defi_20"        : "hypatiax.protocols.experiment_protocol_defi_20",
    "experiment_protocol_nguyen12"       : "hypatiax.protocols.experiment_protocol_nguyen12",
    "experiment_protocol_all_18_a"       : "hypatiax.protocols.experiment_protocol_all_18_a",
    "experiment_protocol_all_20"         : "hypatiax.protocols.experiment_protocol_all_20",
    "experiment_protocol_all_30"         : "hypatiax.protocols.experiment_protocol_all_30",
    "experiment_protocol_benchmark"      : "hypatiax.protocols.experiment_protocol_benchmark",
    "experiment_protocol_benchmark_v2"   : "hypatiax.protocols.experiment_protocol_benchmark_v2",
    "experiment_protocol_comparative"    : "hypatiax.protocols.experiment_protocol_comparative",
    # Support
    "extrapolation_test_protocol"                      : "hypatiax.experiments.tests.extrapolation_test_protocol",
    # Analysis & baselines
    "statistical_analysis"                     : "hypatiax.analysis.statistical_analysis",
    "baseline_neural_network"                          : "hypatiax.core.training.baseline_neural_network",
    "baseline_pure_llm"                                : "hypatiax.core.base_pure_llm.baseline_pure_llm",
}

all_ok = True
for label, dotted in checks.items():
    try:
        importlib.import_module(dotted)
        print(f"  ✓  {label}")
    except Exception as e:
        print(f"  ✗  {label:45s}  ← {e}")
        all_ok = False

print()
print("All imports OK ✓" if all_ok else "☢  Fix errors above before running experiments.")

  ✓  protocols._base
  ✓  protocols.universal_protocol
  ✓  core.runners.common
  ✓  hybrid_system_v50_2
  ✓  symbolic_engine
  ✓  physics_aware_regressor
  ✓  smart_structure_detector
  ✓  dimensional_validator
  ✓  domain_validator
  ✓  ensemble_validator
  ✓  symbolic_validator
  ✓  experiment_protocol_ablation_exp1
  ✓  experiment_protocol_defi_v3
  ✓  experiment_protocol_feynman_exp2
  ✓  experiment_protocol_nguyen12_exp3
  ✓  experiment_protocol_noise_sweep
  ✓  experiment_protocol_hybrid_routing
  ✓  experiment_protocol_instability_rf02_04
  ✓  experiment_protocol_extrapolation_comparative
  ✓  experiment_protocol_provenance_audit
  ✓  experiment_protocol_defi
  ✓  experiment_protocol_defi_20
[juliapkg] Found dependencies: /usr/local/lib/python3.12/dist-packages/pysr/juliapkg.json
[juliapkg] Found dependencies: /usr/local/lib/python3.12/dist-packages/juliapkg/juliapkg.json
[juliapkg] Found dependencies: /usr/local/lib/python3.12/dist-packages/juliacall/juliapkg.json
[juliapkg] L

/usr/local/lib/python3.12/dist-packages/juliacall/__init__.py:61: UserWarning: torch was imported before juliacall. This may cause a segfault. To avoid this, import juliacall before importing torch. For updates, see https://github.com/pytorch/pytorch/issues/78829.
  warnings.warn(


[juliapkg] WARNING: You have Julia 1.12.6 installed but 1.10.3 - 1.11 is required.
[juliapkg]   It is recommended that you upgrade Julia or install JuliaUp.
[juliapkg] Querying Julia versions from https://julialang-s3.julialang.org/bin/versions.json
[juliapkg] WARNING: About to install Julia 1.11.9 to /root/.julia/environments/pyjuliapkg/pyjuliapkg/install.
[juliapkg]   If you use juliapkg in more than one environment, you are likely to
[juliapkg]   have Julia installed in multiple locations. It is recommended to
[juliapkg]   install JuliaUp (https://github.com/JuliaLang/juliaup) or Julia
[juliapkg]   (https://julialang.org/downloads) yourself.
[juliapkg] Downloading Julia from https://julialang-s3.julialang.org/bin/linux/x64/1.11/julia-1.11.9-linux-x86_64.tar.gz
             downloaded 0.1 MB of 273.2 MB
             download complete
[juliapkg] Verifying download
[juliapkg] Installing Julia 1.11.9 to /root/.julia/environments/pyjuliapkg/pyjuliapkg/install
[juliapkg] Using Julia 1.11.

/usr/lib/python3.12/importlib/__init__.py:90: UserWarning: pmlb not installed. Feynman data will be generated analytically.
Install with: pip install pmlb
  return _bootstrap._gcd_import(name[level:], package, level)


  ✓  statistical_analysis
  ✓  baseline_neural_network
  ✓  baseline_pure_llm

All imports OK ✓


## ⚙️ Paper-configuration gate
> **Run this cell before any experiment.** Validates all env vars against `repro.yaml` paper-quality values. Raises `SystemExit` and blocks the run if any value is wrong — most commonly caused by a stale `METHOD_TIMEOUT` or `PYSR_TIMEOUT` from the shell environment overriding the Runtime Config cell.

In [ ]:
# ── Paper-configuration gate — runs before any experiment ────────────────
# Values are sourced from repro.yaml (ground truth).
# If any variable is wrong this cell raises SystemExit and blocks the run.
import os, sys

PAPER_CONFIG = {
    "PYSR_TIMEOUT":   "1100",   # repro.yaml: timeouts.pysr_attempt_seconds
    "METHOD_TIMEOUT": "900",    # repro.yaml: timeouts.method_seconds
    "POPULATIONS":    "30",     # repro.yaml: pysr.populations
    "N_ITERATIONS":   "1000",   # repro.yaml: pysr.niterations
    "NN_SEED":        "42",     # repro.yaml: seeds.default
    "PYSR_SEED":      "42",     # repro.yaml: seeds.pysr_seed
    "LLM_MODEL":      "claude-sonnet-4-20250514",  # repro.yaml: llm_model
}

print("=" * 68)
print("PAPER CONFIGURATION GATE  (repro.yaml v3.0)")
print("=" * 68)

bad = {}
for var, expected in PAPER_CONFIG.items():
    actual = os.environ.get(var)
    ok = actual == expected
    symbol = "\u2713" if ok else ("\u2717" if actual is None else "\u26a0")
    detail = "" if ok else f"  \u2190 expected {expected}" + ("" if actual is None else f", got {actual}")
    print(f"  {symbol} {var}={actual if actual else '(not set)'}{detail}")
    if not ok:
        bad[var] = expected

print("=" * 68)

if bad:
    print("\u274c GATE FAILED — fix these env vars then re-run the Runtime Config cell:")
    for var, val in bad.items():
        print(f"    os.environ[{var!r}] = {val!r}")
    raise SystemExit(
        "\nAbort: paper configuration is wrong. "
        "Run the Runtime Config cell first, or check that no shell env vars "
        "override it (e.g. METHOD_TIMEOUT=1800 from a previous session).\n"
        "The most common cause is running an experiment cell without running "
        "the Runtime Config cell in this session."
    )

print("\u2705 All paper-quality values confirmed — safe to run experiments.")
print("=" * 68)


PAPER CONFIGURATION GATE  (repro.yaml v3.0)
  ✓ PYSR_TIMEOUT=1100
  ✓ METHOD_TIMEOUT=900
  ✓ POPULATIONS=30
  ✓ N_ITERATIONS=1000
  ✓ NN_SEED=42
  ✓ PYSR_SEED=42
  ✓ LLM_MODEL=claude-sonnet-4-20250514
✅ All paper-quality values confirmed — safe to run experiments.


## 5 · Exp 1 — DeFi 74-task benchmark (v3.0)
**Expected:** 89.2 % near-perfect (R²>0.99), 0 catastrophic failures\n**Wall time:** 2–4 h (K=1)


In [ ]:
import os, sys, subprocess
from pathlib import Path

# ── Guard: skip if already completed ────────────────────────────────
try:
    if is_done('exp1'):
        print(f'✅ exp1 already completed — skipping (reset_checkpoint("exp1") to re-run)')
        raise SystemExit(0)
except NameError:
    pass  # checkpoint helpers not loaded — run anyway

# ── Ensure env vars are set (safe to run even if config cell was skipped) ──
os.environ.setdefault('METHOD_TIMEOUT', '900')
os.environ.setdefault('PYSR_TIMEOUT',   '1100')
os.environ.setdefault('PYSR_SEED',      '42')
os.environ.setdefault('NN_SEED',        '42')
os.environ.setdefault('POPULATIONS',    '30')
os.environ.setdefault('N_ITERATIONS',   '1000')

# Force paper-quality timeouts (guard against stale shell env)
os.environ['METHOD_TIMEOUT'] = '900'
os.environ['PYSR_TIMEOUT'] = '1100'

# ── Exp 1 · DeFi 74-task benchmark v3.0 (§10.2–10.4, §10.6) (~3–4 h) ──────────────────────────────────────────────────────
repro = os.environ.get('REPRO_ROOT', '.')
proc = subprocess.Popen(
    [sys.executable, 'run_all_checkpoint.py', '--only', 'exp1', '--pysr-timeout', '1100'],
    cwd=repro,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

if proc.returncode == 0:
    print(f'\n✅ exp1 complete')
    try:
        save_results('exp1')
    except NameError:
        print('  (save_results not loaded — run the backup helper cell)')
else:
    print(f'\n❌ exp1 failed (exit code {proc.returncode})')
    print('   Run: python run_all_checkpoint.py --resume --only exp1')


## 6 · Exp 1b — Portfolio Variance seed sweep  *(§10.5)*
**Expected:** P(HypatiaX > PureLLM) ≈ 0.76 across seeds [42, 99, 123, 777, 2024]  
**Wall time:** ~20–40 min

In [ ]:
import os, sys, subprocess
from pathlib import Path

# ── Guard: skip if already completed ────────────────────────────────
try:
    if is_done('exp1b'):
        print(f'✅ exp1b already completed — skipping (reset_checkpoint("exp1b") to re-run)')
        raise SystemExit(0)
except NameError:
    pass  # checkpoint helpers not loaded — run anyway

# ── Ensure env vars are set (safe to run even if config cell was skipped) ──
os.environ.setdefault('METHOD_TIMEOUT', '900')
os.environ.setdefault('PYSR_TIMEOUT',   '1100')
os.environ.setdefault('PYSR_SEED',      '42')
os.environ.setdefault('NN_SEED',        '42')
os.environ.setdefault('POPULATIONS',    '30')
os.environ.setdefault('N_ITERATIONS',   '1000')

# Force paper-quality timeouts (guard against stale shell env)
os.environ['METHOD_TIMEOUT'] = '900'

# ── Exp 1b · Portfolio Variance seed sweep (§10.5) (~30–60 min) ──────────────────────────────────────────────────────
repro = os.environ.get('REPRO_ROOT', '.')
proc = subprocess.Popen(
    [sys.executable, 'run_all_checkpoint.py', '--only', 'exp1b'],
    cwd=repro,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

if proc.returncode == 0:
    print(f'\n✅ exp1b complete')
    try:
        save_results('exp1b')
    except NameError:
        print('  (save_results not loaded — run the backup helper cell)')
else:
    print(f'\n❌ exp1b failed (exit code {proc.returncode})')
    print('   Run: python run_all_checkpoint.py --resume --only exp1b')


## 7 · Exp 2 — Feynman 30-equation extrapolation  *(§10.7)*
**Expected:** 9/30 (30 %) · 40/60 PCA split · 3 hardware runs (RF-09); Kaggle 4-vCPU is primary  
**Wall time:** 4–8 h  *(skip with `--skip-slow` in `run_all.sh`)*

In [ ]:
import os, sys, subprocess
from pathlib import Path

# ── Guard: skip if already completed ────────────────────────────────
try:
    if is_done('exp2'):
        print(f'✅ exp2 already completed — skipping (reset_checkpoint("exp2") to re-run)')
        raise SystemExit(0)
except NameError:
    pass  # checkpoint helpers not loaded — run anyway

# ── Ensure env vars are set (safe to run even if config cell was skipped) ──
os.environ.setdefault('METHOD_TIMEOUT', '900')
os.environ.setdefault('PYSR_TIMEOUT',   '1100')
os.environ.setdefault('PYSR_SEED',      '42')
os.environ.setdefault('NN_SEED',        '42')
os.environ.setdefault('POPULATIONS',    '30')
os.environ.setdefault('N_ITERATIONS',   '1000')

# Force paper-quality timeouts (guard against stale shell env)
os.environ['METHOD_TIMEOUT'] = '900'
os.environ['PYSR_TIMEOUT'] = '1100'

# ── Exp 2 · Feynman 30-equation extrapolation (§10.7) (~4–8 h) ──────────────────────────────────────────────────────
repro = os.environ.get('REPRO_ROOT', '.')
proc = subprocess.Popen(
    [sys.executable, 'run_all_checkpoint.py', '--only', 'exp2', '--pysr-timeout', '1100'],
    cwd=repro,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

if proc.returncode == 0:
    print(f'\n✅ exp2 complete')
    try:
        save_results('exp2')
    except NameError:
        print('  (save_results not loaded — run the backup helper cell)')
else:
    print(f'\n❌ exp2 failed (exit code {proc.returncode})')
    print('   Run: python run_all_checkpoint.py --resume --only exp2')


## 8 · Exp 3 — Nguyen-12 standard SR suite  *(§10.8 — primary run)*
**Expected:** 11/12 H (91.7 %) · 10/12 P (83.3 %) · 0/12 NN · MW P>NN U=113, p=0.0097  
**Wall time:** 30–90 min  ·  **SEED=42** (fixed for reproducibility)

In [ ]:
import os, sys, subprocess
from pathlib import Path

# ── Guard: skip if already completed ────────────────────────────────
try:
    if is_done('exp3'):
        print(f'✅ exp3 already completed — skipping (reset_checkpoint("exp3") to re-run)')
        raise SystemExit(0)
except NameError:
    pass  # checkpoint helpers not loaded — run anyway

# ── Ensure env vars are set (safe to run even if config cell was skipped) ──
os.environ.setdefault('METHOD_TIMEOUT', '900')
os.environ.setdefault('PYSR_TIMEOUT',   '1100')
os.environ.setdefault('PYSR_SEED',      '42')
os.environ.setdefault('NN_SEED',        '42')
os.environ.setdefault('POPULATIONS',    '30')
os.environ.setdefault('N_ITERATIONS',   '1000')

# Force paper-quality timeouts (guard against stale shell env)
os.environ['METHOD_TIMEOUT'] = '900'
os.environ['PYSR_TIMEOUT'] = '1100'

# ── Exp 3 · Nguyen-12 SEED=42 (§10.8 primary) (~30–90 min) ──────────────────────────────────────────────────────
repro = os.environ.get('REPRO_ROOT', '.')
proc = subprocess.Popen(
    [sys.executable, 'run_all_checkpoint.py', '--only', 'exp3', '--pysr-timeout', '1100'],
    cwd=repro,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

if proc.returncode == 0:
    print(f'\n✅ exp3 complete')
    try:
        save_results('exp3')
    except NameError:
        print('  (save_results not loaded — run the backup helper cell)')
else:
    print(f'\n❌ exp3 failed (exit code {proc.returncode})')
    print('   Run: python run_all_checkpoint.py --resume --only exp3')


### 8b · Exp 3b — Nguyen-12 stability check  *(§10.8 — seeds 99/123/777/2024)*
**Expected:** consistent with SEED=42 across all 5 seeds  
**Wall time:** 30–90 min

In [ ]:
import os, sys, subprocess
from pathlib import Path

# ── Guard: skip if already completed ────────────────────────────────
try:
    if is_done('exp3b'):
        print(f'✅ exp3b already completed — skipping (reset_checkpoint("exp3b") to re-run)')
        raise SystemExit(0)
except NameError:
    pass  # checkpoint helpers not loaded — run anyway

# ── Ensure env vars are set (safe to run even if config cell was skipped) ──
os.environ.setdefault('METHOD_TIMEOUT', '900')
os.environ.setdefault('PYSR_TIMEOUT',   '1100')
os.environ.setdefault('PYSR_SEED',      '42')
os.environ.setdefault('NN_SEED',        '42')
os.environ.setdefault('POPULATIONS',    '30')
os.environ.setdefault('N_ITERATIONS',   '1000')

# Force paper-quality timeouts (guard against stale shell env)
os.environ['METHOD_TIMEOUT'] = '900'
os.environ['PYSR_TIMEOUT'] = '1100'

# ── Exp 3b · Nguyen-12 seeds 99/123/777/2024 (§10.8) (~30–90 min) ──────────────────────────────────────────────────────
repro = os.environ.get('REPRO_ROOT', '.')
proc = subprocess.Popen(
    [sys.executable, 'run_all_checkpoint.py', '--only', 'exp3b', '--pysr-timeout', '1100'],
    cwd=repro,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

if proc.returncode == 0:
    print(f'\n✅ exp3b complete')
    try:
        save_results('exp3b')
    except NameError:
        print('  (save_results not loaded — run the backup helper cell)')
else:
    print(f'\n❌ exp3b failed (exit code {proc.returncode})')
    print('   Run: python run_all_checkpoint.py --resume --only exp3b')


## 9 · Supp B — Noise & sample-complexity sweep
**Expected:** EHD 100 % at all σ levels; plateau ~N=500\n**Wall time:** 6–12 h


In [ ]:
import os, sys, subprocess
from pathlib import Path

# ── Guard: skip if already completed ────────────────────────────────
try:
    if is_done('suppB'):
        print(f'✅ suppB already completed — skipping (reset_checkpoint("suppB") to re-run)')
        raise SystemExit(0)
except NameError:
    pass  # checkpoint helpers not loaded — run anyway

# ── Ensure env vars are set (safe to run even if config cell was skipped) ──
os.environ.setdefault('METHOD_TIMEOUT', '900')
os.environ.setdefault('PYSR_TIMEOUT',   '1100')
os.environ.setdefault('PYSR_SEED',      '42')
os.environ.setdefault('NN_SEED',        '42')
os.environ.setdefault('POPULATIONS',    '30')
os.environ.setdefault('N_ITERATIONS',   '1000')

# Force paper-quality timeouts (guard against stale shell env)
os.environ['METHOD_TIMEOUT'] = '900'

# ── Supp B · Noise & sample-complexity sweep [SLOW] (~6–12 h) ──────────────────────────────────────────────────────
repro = os.environ.get('REPRO_ROOT', '.')
proc = subprocess.Popen(
    [sys.executable, 'run_all_checkpoint.py', '--only', 'suppB'],
    cwd=repro,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

if proc.returncode == 0:
    print(f'\n✅ suppB complete')
    try:
        save_results('suppB')
    except NameError:
        print('  (save_results not loaded — run the backup helper cell)')
else:
    print(f'\n❌ suppB failed (exit code {proc.returncode})')
    print('   Run: python run_all_checkpoint.py --resume --only suppB')


## 10 · Supp A — Hybrid routing improvements  *(Supp A §1–8)*
**Expected:** +6 pp Fix1 (extrapolation probe) · +5 pp Fix2 (formula complexity) · +1 pp Fix3 (LLM feature aug)  
Override priority: Fix2 > Fix1 > Fix3 > Fix4  
**Wall time:** ~30–60 min

In [ ]:
import os, sys, subprocess
from pathlib import Path

# ── Guard: skip if already completed ────────────────────────────────
try:
    if is_done('suppA'):
        print(f'✅ suppA already completed — skipping (reset_checkpoint("suppA") to re-run)')
        raise SystemExit(0)
except NameError:
    pass  # checkpoint helpers not loaded — run anyway

# ── Ensure env vars are set (safe to run even if config cell was skipped) ──
os.environ.setdefault('METHOD_TIMEOUT', '900')
os.environ.setdefault('PYSR_TIMEOUT',   '1100')
os.environ.setdefault('PYSR_SEED',      '42')
os.environ.setdefault('NN_SEED',        '42')
os.environ.setdefault('POPULATIONS',    '30')
os.environ.setdefault('N_ITERATIONS',   '1000')

# Force paper-quality timeouts (guard against stale shell env)
os.environ['METHOD_TIMEOUT'] = '900'

# ── Supp A · Hybrid routing improvements (Fix 1–5b) (~30–60 min) ──────────────────────────────────────────────────────
repro = os.environ.get('REPRO_ROOT', '.')
proc = subprocess.Popen(
    [sys.executable, 'run_all_checkpoint.py', '--only', 'suppA'],
    cwd=repro,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

if proc.returncode == 0:
    print(f'\n✅ suppA complete')
    try:
        save_results('suppA')
    except NameError:
        print('  (save_results not loaded — run the backup helper cell)')
else:
    print(f'\n❌ suppA failed (exit code {proc.returncode})')
    print('   Run: python run_all_checkpoint.py --resume --only suppA')


## 11 · §10.9 — Instability analysis (RF02/04)
**Expected:** Spearman ρ=−0.70 (p<0.001) · 70 tasks × K=30 runs · C-Collapse Portfolio ES anomaly (RF-06)  
**Wall time:** 3–6 h  (K=30 full sweep)  

> ⚠️ **The cell below sets `LLM_K_RUNS=30` for the full sweep.**  
> Set `LLM_K_RUNS=1` in cell 2 (Runtime config) for a smoke-test only.

In [ ]:
import os, sys, subprocess
from pathlib import Path

# ── Guard: skip if already completed ────────────────────────────────
try:
    if is_done('instability'):
        print(f'✅ instability already completed — skipping (reset_checkpoint("instability") to re-run)')
        raise SystemExit(0)
except NameError:
    pass  # checkpoint helpers not loaded — run anyway

# ── Ensure env vars are set (safe to run even if config cell was skipped) ──
os.environ.setdefault('METHOD_TIMEOUT', '900')
os.environ.setdefault('PYSR_TIMEOUT',   '1100')
os.environ.setdefault('PYSR_SEED',      '42')
os.environ.setdefault('NN_SEED',        '42')
os.environ.setdefault('POPULATIONS',    '30')
os.environ.setdefault('N_ITERATIONS',   '1000')

# Force paper-quality timeouts (guard against stale shell env)
os.environ['METHOD_TIMEOUT'] = '900'
os.environ['PYSR_TIMEOUT'] = '1100'

# ── §10.9 · Instability analysis K=30 [SLOW] (~3–6 h) ──────────────────────────────────────────────────────
repro = os.environ.get('REPRO_ROOT', '.')
proc = subprocess.Popen(
    [sys.executable, 'run_all_checkpoint.py', '--only', 'instability', '--pysr-timeout', '1100'],
    cwd=repro,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

if proc.returncode == 0:
    print(f'\n✅ instability complete')
    try:
        save_results('instability')
    except NameError:
        print('  (save_results not loaded — run the backup helper cell)')
else:
    print(f'\n❌ instability failed (exit code {proc.returncode})')
    print('   Run: python run_all_checkpoint.py --resume --only instability')


## 12 · §10.8 — Extrapolation comparative
**Expected:** cross-method R² comparison across near/medium/far OOD regimes  
**Wall time:** ~20–40 min

In [ ]:
import os, sys, subprocess
from pathlib import Path

# ── Guard: skip if already completed ────────────────────────────────
try:
    if is_done('extrap'):
        print(f'✅ extrap already completed — skipping (reset_checkpoint("extrap") to re-run)')
        raise SystemExit(0)
except NameError:
    pass  # checkpoint helpers not loaded — run anyway

# ── Ensure env vars are set (safe to run even if config cell was skipped) ──
os.environ.setdefault('METHOD_TIMEOUT', '900')
os.environ.setdefault('PYSR_TIMEOUT',   '1100')
os.environ.setdefault('PYSR_SEED',      '42')
os.environ.setdefault('NN_SEED',        '42')
os.environ.setdefault('POPULATIONS',    '30')
os.environ.setdefault('N_ITERATIONS',   '1000')

# Force paper-quality timeouts (guard against stale shell env)
os.environ['METHOD_TIMEOUT'] = '900'

# ── §10.8 · Extrapolation comparative (~20–40 min) ──────────────────────────────────────────────────────
repro = os.environ.get('REPRO_ROOT', '.')
proc = subprocess.Popen(
    [sys.executable, 'run_all_checkpoint.py', '--only', 'extrap'],
    cwd=repro,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

if proc.returncode == 0:
    print(f'\n✅ extrap complete')
    try:
        save_results('extrap')
    except NameError:
        print('  (save_results not loaded — run the backup helper cell)')
else:
    print(f'\n❌ extrap failed (exit code {proc.returncode})')
    print('   Run: python run_all_checkpoint.py --resume --only extrap')


## 📥 Download Results
Download all experiment outputs (JSON data, figures, tables, logs) as a single zip.  
Works on **Colab**, **Kaggle**, and **local**. Run after experiments complete.

In [ ]:
# ── Download all results ──────────────────────────────────────────────────────
# Zips hypatiax/data/results/ + logs/ and downloads to your machine.
# Colab  : triggers browser download automatically
# Kaggle : copies zip to /kaggle/working/ (visible in Output tab)
# Local  : saves zip next to the repo, prints the path
import os, shutil, pathlib, datetime, sys

REPRO_ROOT  = pathlib.Path(os.environ.get('REPRO_ROOT', '.')).resolve()
RESULTS_DIR = REPRO_ROOT / 'hypatiax' / 'data' / 'results'
LOGS_DIR    = REPRO_ROOT / 'logs'

# ── Timestamp for unique filename ─────────────────────────────────────────────
ts       = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
zip_name = f'hypatiaX_results_{ts}'

# ── Build a staging folder with everything worth keeping ──────────────────────
staging  = pathlib.Path('/tmp') / zip_name
staging.mkdir(parents=True, exist_ok=True)

def _copy(src, dst_name):
    src = pathlib.Path(src)
    if src.exists():
        dst = staging / dst_name
        if src.is_dir():
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst)
        print(f'  ✓  {dst_name}')
    else:
        print(f'  ⚠  {dst_name} not found — skipping')

print('Staging results ...')
_copy(RESULTS_DIR,                           'results')          # all JSON + figures + tables
_copy(LOGS_DIR,                              'logs')             # per-step logs + provenance
_copy(REPRO_ROOT / 'logs' / 'pipeline_checkpoint.json',
      'pipeline_checkpoint.json')                                 # resume checkpoint

# ── Create zip ────────────────────────────────────────────────────────────────
zip_path = pathlib.Path('/tmp') / zip_name
final_zip = shutil.make_archive(str(zip_path), 'zip', root_dir='/tmp', base_dir=zip_name)
size_mb   = pathlib.Path(final_zip).stat().st_size / 1024**2
print(f'\nZip created: {final_zip}  ({size_mb:.1f} MB)')

# ── Platform-aware download ───────────────────────────────────────────────────
IS_COLAB  = 'google.colab' in sys.modules or os.path.exists('/content')
IS_KAGGLE = os.path.exists('/kaggle')

if IS_COLAB:
    from google.colab import files
    print('Downloading via Colab ...')
    files.download(final_zip)
    print('✓ Download triggered — check your browser Downloads folder.')

elif IS_KAGGLE:
    kaggle_out = pathlib.Path('/kaggle/working') / pathlib.Path(final_zip).name
    shutil.copy2(final_zip, kaggle_out)
    print(f'✓ Kaggle: zip saved to Output tab → {kaggle_out.name}')
    print('  Open the Output tab (right panel) and click the file to download.')

else:
    local_out = REPRO_ROOT.parent / pathlib.Path(final_zip).name
    shutil.copy2(final_zip, local_out)
    print(f'✓ Local: zip saved to {local_out}')

# ── Contents summary ─────────────────────────────────────────────────────────
print('\n── Contents ─────────────────────────────────────────────────────────')
n_json = sum(1 for _ in RESULTS_DIR.rglob('*.json')) if RESULTS_DIR.exists() else 0
n_pdf  = sum(1 for _ in (RESULTS_DIR / 'figures').glob('*.pdf')) if (RESULTS_DIR / 'figures').exists() else 0
n_tex  = sum(1 for _ in (RESULTS_DIR / 'tables').glob('*.tex'))  if (RESULTS_DIR / 'tables').exists()  else 0
n_log  = sum(1 for _ in LOGS_DIR.rglob('*.log'))                 if LOGS_DIR.exists() else 0
print(f'  JSON result files : {n_json}')
print(f'  Figures (PDF)     : {n_pdf}')
print(f'  Tables  (TeX)     : {n_tex}')
print(f'  Log files         : {n_log}')
print(f'  Zip size          : {size_mb:.1f} MB')


## 13 · §11 — Provenance audit
**Expected:** Run after all experiments complete


In [ ]:
# ── Provenance audit (§11) — non-blocking ────────────────────────────────
import os, sys, subprocess
repro = os.environ.get('REPRO_ROOT', '.')
proc = subprocess.Popen(
    [sys.executable, 'run_all_checkpoint.py', '--only', 'provenance'],
    cwd=repro,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f'\n✅ Provenance audit done' if proc.returncode == 0 else f'\n⚠️  Provenance audit exited {proc.returncode} (non-blocking)')


In [ ]:
# §11b — Link every result file to its source family and paper section
# Reads provenance_map.json (optional); outputs logs/provenance_audit/
import pathlib, os

REPO_BASE = pathlib.Path(os.environ["REPRO_ROOT"])
pmap      = REPO_BASE / "provenance_map.json"

if not pmap.exists():
    print(f"⚠  provenance_map.json not found at {pmap} — skipping §11b.")
    print("   To generate it, run:  python audit/discover_provenance.py --generate-map")
else:
    out_dir = REPO_BASE / "logs" / "provenance_audit"
    out_dir.mkdir(parents=True, exist_ok=True)
    !python audit/discover_provenance.py --root {str(REPO_BASE)} --map {str(pmap)} --out {str(out_dir)}
    if _exit_code != 0:
        print("  ⚠  audit/discover_provenance.py exited non-zero — check output above")


In [ ]:
# §11c — Build internal import DAG for reproducibility verification
# Outputs logs/repro_output/import_graph.{dot,png,pdf}
import pathlib, os

REPO_BASE = pathlib.Path(os.environ["REPRO_ROOT"])
out_dir   = REPO_BASE / "logs" / "repro_output"
out_dir.mkdir(parents=True, exist_ok=True)

!python audit/scan_internal_imports.py --root {str(REPO_BASE)} --out {str(out_dir)}
# Import graph rendered to logs/repro_output/import_graph.{dot,png,pdf}


## 14 · Validate metrics against paper

| Metric | Paper target | Section |
|---|---|---|
| R²>0.99 rate (DeFi 74-task)          | 89.2 %       | §10.2 |
| Catastrophic failures                | 0            | §10.2 |
| Speedup (LLM-routed cases)           | 1.73×        | §10.4 |
| Portfolio Variance P(H>P)            | ≈ 0.76       | §10.5 |
| Ablation MW U (Run A)                | 126, p=0.295 | §10.6 |
| Feynman success rate                 | 9/30 (30 %)  | §10.7 |
| Nguyen-12 HypatiaX rate              | 11/12 (91.7%)| §10.8 |
| Instability Spearman ρ               | −0.70, p<0.001| §10.9 |

In [ ]:
import os, sys, subprocess
repro = os.environ.get('REPRO_ROOT', '.')

def run_streaming(cmd):
    proc = subprocess.Popen(cmd, cwd=repro, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    return proc.returncode

rc1 = run_streaming([sys.executable, 'scripts/patches/verify_results.py', '--report', '--json'])
rc2 = run_streaming([sys.executable, 'hypatiax/reproducibility/hash_lock.py', '--check'])

if rc1 != 0 or rc2 != 0:
    raise RuntimeError('Verification failed')
print('\n✓ Verification done')


## 15 · Generate figures and tables


In [ ]:
import os, pathlib
_cwd = os.environ.get('REPRO_ROOT', '.')
_results = str(pathlib.Path(_cwd) / 'hypatiax' / 'data' / 'results')
_tbl_dir = str(pathlib.Path(_results) / 'tables')

# Figures → hypatiax/data/results/figures/
!python figures/generate_figures.py --outdir {_results}/figures
if _exit_code != 0:
    print('  ⚠ generate_figures.py failed (non-blocking)')

# Tables → hypatiax/data/results/tables/
# v7: FIX-TABLES — pass TABLE_OUTDIR + VERIFY_RESULTS_DIR so generate_tables.py
# writes to the correct canonical location (not paper/tables/).
%env TABLE_OUTDIR={_tbl_dir}
%env VERIFY_RESULTS_DIR={_results}
!python scripts/patches/generate_tables.py --outdir {_tbl_dir}
if _exit_code != 0:
    print('  ⚠ generate_tables.py failed (non-blocking)')

# Report counts
fig_dir = pathlib.Path(_results) / 'figures'
tbl_dir = pathlib.Path(_results) / 'tables'
# v7: FIX-INVENTORY — also check paper/tables/ as fallback
if not tbl_dir.exists() or not any(tbl_dir.glob('*.tex')):
    tbl_dir = pathlib.Path(_cwd) / 'paper' / 'tables'
n_pdf = len(list(fig_dir.glob('*.pdf'))) if fig_dir.exists() else 0
n_tex = len(list(tbl_dir.glob('*.tex'))) if tbl_dir.exists() else 0
print(f'✓ Figures: {n_pdf} PDF(s) in {fig_dir}')
print(f'✓ Tables : {n_tex} TeX(s) in {tbl_dir}')


## 📥 Download Results
Download all experiment outputs (JSON data, figures, tables, logs) as a single zip.  
Works on **Colab**, **Kaggle**, and **local**. Run after experiments complete.